# RegDet V1.1 — FIT STABILITY (S) vs TRANSITION LAG (L)

Implements `STABILITY_LAG_PROTOCOL.md` **exactly**. The protocol was frozen before
any number in this notebook existed.

## The two defects

**D1 — fit instability.** Refitting the same arm with a *disjoint seed set* flips
~6% of all bars. That is larger than the entire 25%↔100% HMM-share difference, so
it contaminates every comparison in the project.

**D2 — transition lag.** Labels describe the regime that *just ended*. Reference
case: the May-2025 V-bottom, labelled `L_BEAR` for the whole +2.7% advance and
turning `BULL` only near the top.

## The central tension — stated up front, not hidden

> Lower lag = react sooner = react to smaller moves = **more** switches and **more**
> sensitivity to the underlying fit. **These objectives oppose each other.** The
> honest deliverable may be a FRONTIER, not a fix. If nothing improves one without
> degrading the other, this notebook prints `NO DOMINATING CONFIGURATION` and shows
> the frontier. It does not manufacture a winner.

## The three metrics (frozen)

| | definition | direction |
|---|---|---|
| **S** | mean pairwise 5-label agreement across **R = 4 disjoint seed sets** | higher better |
| **L** | bars from a label-blind 2.0% ZigZag swing start to the first correctly-directed emitted label; unmatched swings score the **full swing length** | lower better |
| **W** | label switches per 100 bars | guard — must not get worse |

## Guards — a candidate is VOID if any fails

`G1` every label occupancy in [3%, 50%] · `G2` no label collapses below 3% ·
`G3` W ≤ 25 / 100 bars · `G4` SIDEWAYS ≤ 50%.

**G4 exists because S is trivially maximised by labelling everything SIDEWAYS.**
Section 3 drives that exact degenerate case through the real guard code and asserts
it is VOIDED — the guard is tested, not trusted.

## Fixed by decree

`BAR_DIR_WEIGHT = 0.0` throughout. It is settled and is **not** swept here; the
notebook asserts it at every labelling call.

In [ ]:
%pip install -q hmmlearn yfinance

## 1. Engine, inlined verbatim

Inlined out of `build_master_notebook_v2.py` rather than imported, because a
standalone `.ipynb` on Kaggle cannot import a local `.py` sitting next to it.
Source: `build_master_notebook_v2.py cell 3 (constants/imports/CONFIGS), cell 5 (load_2h/_synth), cell 7 (feature + labeling engine), cell 9 (regime-background plot helpers)`.

Not one line is modified — same constants, same features, same gates, same
hysteresis, same causality proofs. The master notebook, its generator,
`regime_scorecard.py` and `build_hmm_share_charts.py` are untouched by this
notebook.

In [ ]:
# ==========================================================================
# CONSTANTS + IMPORTS -- INLINED VERBATIM from build_master_notebook_v2.py
# (master notebook code cell 3). Unmodified.
# ==========================================================================
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt          # default backend -> inline figures
import matplotlib.patches as mpatches
import matplotlib.dates as mdates        # date2num for the batched regime bands
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

# ---------------------------------------------------------------------------
# Walk-forward harness config (matches capacity_ladder.py / the head-to-head)
# ---------------------------------------------------------------------------
HMM_ITER       = 2000          # EM iteration cap (fits converge in ~75-240 iters, <1s each)
N_FOLDS        = 4             # anchored walk-forward test blocks
MIN_TRAIN_FRAC = 0.50          # first fold trains on >= this fraction of bars
BARS_PER_DAY   = 3             # 2h bars per NSE session
LOOKBACK_SCALE = 1.0           # feature-window multiplier (1.0 = production windows)

# ---------------------------------------------------------------------------
# LABELING KNOBS — verbatim from REGDET_CONFIG in regdet_v11.py.
# These drive the direction + intensity + chop-filter scheme used EVERYWHERE in
# this notebook (walk-forward signal, regime overlay, evaluation suite).
# ---------------------------------------------------------------------------
CONF_L         = 0.50   # low-confidence override: if the WINNING direction bucket's
                        # aggregated probability mass is below this, the bar is forced
                        # to SIDEWAYS. Higher = more SIDEWAYS, fewer directional calls.
Z_HI           = 0.5    # trend-MAGNITUDE gate: |trend_z| must reach this for a bull/bear
                        # bar to escalate to H_BULL/H_BEAR. trend_z is TREND_FEATURE
                        # standardized against a mu/sd baseline FROZEN on the fit window
                        # (causal — never recomputed from bars the model has not seen).
                        # Lower = H fires more often (and flickers more).
EFF_HI         = 0.35   # trend-EFFICIENCY gate = THE CHOP FILTER. Efficiency is
                        # |net move| / |total path walked| over EFF_WIN bars: ~1.0 in a
                        # clean trend, ~0.0 in a range that keeps doubling back. Magnitude
                        # alone cannot separate a real trend from a swing inside a bracket
                        # (both show big momentum); this can. Requiring BOTH gates keeps
                        # range-bound swings at L_BULL/L_BEAR. Raise = stricter (less H).
EFF_WIN        = 9      # bars over which efficiency is measured (~ TREND_FEATURE horizon)
TREND_FEATURE  = 'mom_3d'   # the feature whose z-score grades trend magnitude

# ---------------------------------------------------------------------------
# LABELING FIXES 1-3 (see Section 7H for the measured A/B). Each is behind its
# own switch, and each switch has a value that reproduces the OLD behaviour
# bit-for-bit, so their effects can be separated and audited.
# ---------------------------------------------------------------------------

# FIX 1 -- DIRECTION SCORER: features that are excluded from the composite
# BULLISHNESS score. vol_2h and vol_expansion are MAGNITUDE measures: they say
# how big moves are, not which way they point. Feeding them into a DIRECTION
# score is a category error -- with the old weights the risk block (vol_2h,
# vol_expansion, vix_chg, drawdown; total 4.0) outweighed the directional block
# (ret_2h, mom_1d/3d/5d, dist_ma; total 3.2), so a violent RALLY (high vol, high
# vol expansion, still-elevated drawdown) scored BEARISH while price ripped up.
# These two features REMAIN HMM INPUT FEATURES -- they are genuinely informative
# about state -- they simply stop voting on direction. drawdown and vix_chg keep
# their -1.0 direction weight: both are defensibly signed (a deeper drawdown and
# a rising VIX really are bearish, not merely "big").
# DIRECTION_EXCLUDE = () reproduces the old scorer exactly.
DIRECTION_EXCLUDE = ('vol_2h', 'vol_expansion')

# FIX 2 -- LABEL HYSTERESIS (causal confirmation delay): a candidate new
# DIRECTION must persist for CONFIRM_BARS consecutive bars before the emitted
# label is allowed to flip; until then the previous emitted direction is held.
# This is a DELAY, not a smoother: bar t's emitted label is a function of bars
# <= t only. (An earlier version of this project shipped a "smoother" that
# decided whether to erase a run by inspecting the run's full REALIZED length --
# that is look-ahead and was removed. Nothing of that shape is reintroduced here;
# the causality probe in 7.0 re-proves it under hysteresis.)
# CONFIRM_BARS = 1 reproduces the old behaviour exactly.
CONFIRM_BARS   = 2

# FIX 3 -- GATE HYSTERESIS: the H escalation gates become enter/exit BANDS.
# Escalate L -> H when |z| >= Z_HI AND eff >= EFF_HI (unchanged), but only
# de-escalate H -> L when |z| < Z_HI_EXIT OR eff < EFF_HI_EXIT. Without this,
# bars sitting near a single threshold flicker H->L->H->L every bar.
# Z_HI_EXIT = Z_HI and EFF_HI_EXIT = EFF_HI reproduces the old behaviour exactly.
Z_HI_EXIT      = 0.35
EFF_HI_EXIT    = 0.25

# FIX 4 -- PER-BAR DIRECTION (this is the architectural one).
#
# THE FLAW IT ADDRESSES. Until now DIRECTION was a per-STATE property: the HMM
# assigns bar t a distribution over states, each STATE is bucketed bear/side/bull
# ONCE from the rank of its composite mean profile, and the bar inherits its
# state's direction. But an HMM state is directionally MIXED. Volatility is
# direction-agnostic, so the model reliably learns a "violent" state that
# contains BOTH sharp selloffs AND sharp rallies. ONE bucket label for that state
# cannot be right for every bar in it, no matter how the bucket is chosen.
#
# Real-data evidence (2874 2h Nifty bars) on the 366 causal high-vol RALLY bars
# (top vol_2h quartile AND trailing mom_3d > 0):
#   baseline          46.2% bear / 7.4% sideways / 46.4% bull
#   + FIX 1 (no vol)  11.2% bear / 42.3% sideways / 46.4% bull
# The red became GREY, not green -- bull did not move at all -- and SIDEWAYS then
# showed the HIGHEST forward return of any label (+0.717% at 15 bars vs H_BULL
# +0.026%), which is exactly what "a bullish population got parked in SIDEWAYS"
# looks like. FIX 1 removed a wrong vote; it could not add a right one, because
# the vote is cast once per state and not once per bar.
#
# THE FIX. Add a CAUSAL PER-BAR directional score from the SIGNED features only
# (ret_2h, mom_1d, mom_3d, mom_5d, dist_ma -- reusing FEATURE_SIGN/FEATURE_MAG),
# each standardized against a mu/sd baseline FROZEN on the fit window exactly the
# way trend_z is, then map it to three per-bar direction masses through a softmax
# and BLEND those with the state-level masses:
#
#     mass = (1 - BAR_DIR_WEIGHT) * state_mass + BAR_DIR_WEIGHT * bar_mass
#
# A convex combination of two points on the 3-simplex is on the 3-simplex, so
# bull+side+bear == 1 still holds bar by bar and every downstream mechanism --
# the prob_* partition, tactical_regime_confidence, the CONF_L override, the
# intensity gates, the CONFIRM_BARS hysteresis -- is untouched. The HMM keeps
# supplying market character, persistence and the confidence signal; only the
# DIRECTION ATTRIBUTION moves from per-state to per-bar.
#
# vol_2h / vol_expansion are structurally excluded from this score: they are
# magnitude, not direction. That is asserted, not merely intended.
#
# BAR_DIR_WEIGHT = 0.0 reproduces the pre-fix behaviour BIT-FOR-BIT -- the blend
# degenerates to 1.0*state_mass + 0.0*bar_mass, which is exact in IEEE754 for
# non-negative masses. Section 7H asserts that equality rather than assuming it.
# BAR_DIR_WEIGHT = 1.0 decides direction purely per bar (the HMM then contributes
# only character/persistence, not direction). Section 7H sweeps 0/0.25/0.5/0.75/1.
#
# ADOPTED VALUE 0.0 -- FIX 4 IS RETAINED AS A SWITCH BUT SET OFF.
#
# It was 0.75 (and before that 0.5). It is now 0.0. The machinery, the sweep in
# 7H-vi and the overlay in 7H-vii all stay; only the shipped weight moved.
#
# WHY IT WAS TURNED OFF. Scored across three real-data runs, the benefit fix 4 was
# added for -- more bull on high-volatility rally bars -- did NOT reproduce:
# +19.7 pp once, +1.4 pp the second time, and on the third the rally gain came
# from fixes 1-2 with fix 4 already OFF. The COST reproduced every time: it broke
# the direction-level forward-return ordering. A controlled A/B on the SAME data
# and the SAME fit, changing only w:
#
#     metric                        w = 0.75        w = 0.0
#     direction ordering 3/9/15     BROKEN/HOLDS/BROKEN   HOLDS/HOLDS/HOLDS
#     H_BULL fwd_3 HAC t            1.93 WARN       2.06 PASS
#     L_BULL fwd_9 HAC t            1.04 FAIL       2.10 PASS
#     BULL vs BEAR d @ fwd_3        0.006           0.118
#     H_BULL vs H_BEAR d @ fwd_3    0.139 WARN      0.214 PASS
#     strategy total return         +17.37%         +37.74%
#     strategy Sharpe               0.69            1.26
#
# w = 0 is where this project first produced HAC t > 2 on anything.
#
# The dead-zone note that justified 0.75 over 0.5 is still true and still the
# reason NOT to ship an intermediate value if fix 4 is ever switched back on: the
# filtered state posterior saturates near one-hot (confidence ~0.999 on most
# bars), so a convex blend cannot move the argmax until w > ~0.57. The choice is
# effectively between 0.0 (off) and >= ~0.75 (on); 0.25/0.5 are nominally on and
# behaviourally almost off, which is the worst of both.
#
# At w = 0.0 `dir_feats` is STILL passed everywhere it was before. The blend
# degenerates to 1.0*state_mass + 0.0*bar_mass -- exact in IEEE754 -- so labels
# are bit-identical to the no-dir_feats path (asserted in 7H), while
# `bar_dir_score` stays populated as a diagnostic column and the causality probe
# in Section 7.0 keeps testing it.
BAR_DIR_WEIGHT   = 0.0
BAR_DIR_FEATURES = ('ret_2h', 'mom_1d', 'mom_3d', 'mom_5d', 'dist_ma')
BAR_DIR_TAU      = 1.0   # softmax temperature on the per-bar z. The score is
                         # re-standardized on the fit window, so tau is in units
                         # of fit-window sd: |z| ~ 0.5*tau is where the leading
                         # direction's per-bar mass crosses 0.5. Lower tau =
                         # more decisive (more extreme) per-bar masses.

TRAIN_FRACTION = 0.70   # anchored fit fraction used by the PRODUCTION-style single fit
                        # in Section 7 (the engine's own TRAIN_FRACTION). The walk-forward
                        # in Section 5 uses MIN_TRAIN_FRAC/N_FOLDS fold edges instead.

# ---------------------------------------------------------------------------
# SEED ENSEMBLE — the identifiability fix (see Section 5d).
#
# hmm.GaussianHMM is fit by EM, a LOCAL optimizer. With one fixed seed the fit
# is not identified: refitting the same bars with different seeds lands in
# different local optima (train log-likelihood spread of ~1000 nats across 8
# seeds on the full-cov config) which segment the data differently. Multi-restart
# EM keeping the best log-likelihood does NOT fix it -- it collapses the LL
# spread but the surviving optima still disagree on the segmentation.
#
# What works instead: fit K models with K different seeds and average the
# DIRECTION-BUCKET PROBABILITY MASSES (bull / side / bear). Raw HMM state indices
# are arbitrary and permute freely between fits, so they cannot be averaged --
# but direction masses are permutation-INVARIANT semantic quantities, so they
# can. Each model's 3 masses sum to 1, so their average does too, and the
# labeling scheme downstream is bit-for-bit the same; only the SOURCE of the
# masses changes.
#
# ENSEMBLE_K = 1 reproduces the old single-fit behaviour exactly.
#
# Cost scales linearly in K (K fits per training slice). K=6 was the size the
# offline study measured (direction-call agreement between independent pools
# 81.9% single -> 91.9% at K=6), and K=6 is now the ADOPTED value: it is the size
# the stability study actually measured, and the cost is linear. The earlier K=4
# compromise existed only because Section 5d re-ran the ENTIRE walk-forward in
# both arms; 5d is OFF by default now (RUN_SEED_STABILITY=False below), so the
# runtime argument for K=4 no longer applies.
ENSEMBLE_K     = 6
BASE_SEED      = 42     # ensemble seeds are BASE_SEED + 0..K-1 (deterministic,
                        # so every run of this notebook is reproducible)

# ---------------------------------------------------------------------------
# RUNTIME KNOBS. These change ONLY how fast the notebook runs, never what it
# computes. Both have a value that reproduces the original code path exactly.
# ---------------------------------------------------------------------------

# N_JOBS -- ensemble fit parallelism. DEFAULT 1 (serial), and that default is
# deliberate. Read this before changing it.
#
# The K members of a seed ensemble ARE independent and each IS fully determined
# by its own random_state, so at the level of the algorithm, fitting them
# concurrently cannot change anything. It does anyway, for a reason that has
# nothing to do with this notebook's logic: MEASURED on this environment,
# `GaussianHMM.fit` is not bit-reproducible across OpenBLAS thread counts. The
# same seed on the same rows gives model parameters differing by ~1.5e-11
# between a 1-thread and a 4-thread BLAS, because threaded reductions sum in a
# different order and ~100-170 EM iterations amplify the last-bit difference.
# joblib's loky backend pins each worker to ONE inner thread (correctly -- it is
# avoiding oversubscription), so a parallel fit lands on the 1-thread arithmetic
# while the serial fit here lands on the multi-thread arithmetic.
#
# Measured, on 4 fits of 1500x9 at N=5 full-cov:
#     serial, default threads          3.19s   <- what this notebook does
#     serial, BLAS pinned to 1 thread  2.88s   params differ by 1.5e-11
#     parallel, 1 inner thread         2.48s   BIT-IDENTICAL to the line above
#     parallel, 4 inner threads       33.64s   10x SLOWER (oversubscription)
#
# So parallelism is available but only at 1 inner thread, and that arm is
# bit-identical to serial-at-1-thread -- NOT to serial-at-default-threads. The
# available speedup is ~1.3x on the fits, and the price is moving every model
# parameter in the 11th decimal. In a notebook where a 3-bar data perturbation
# has already flipped the config winner, that is a bad trade, so it is NOT the
# default. N_JOBS = 1 reproduces the historical numbers exactly.
#
# If you set N_JOBS != 1 you are choosing a different (equally valid, not more
# accurate) floating-point path, and the headline numbers may move slightly.
# Section 7.0 asserts and REPORTS this rather than hiding it.
N_JOBS         = 1

# Section 5d is a ONE-TIME IDENTIFIABILITY DIAGNOSTIC, not part of the pipeline:
# it re-runs the ENTIRE walk-forward 36 times (3 configs x 6 seeds x 2 arms) to
# ask whether the seed ensemble stabilised the config ranking. That question has
# been answered, and NOTHING downstream reads any variable it defines -- so on a
# normal run it is ~2/3 of the total wall time spent re-confirming a settled
# result. Default OFF. Set True to re-run it (e.g. after changing ENSEMBLE_K,
# N_STATES, the features or the folds -- any of which reopens the question).
RUN_SEED_STABILITY = False

CONFIDENCE_THRESHOLD_H  = 0.70   # chart reference line only
CONFIDENCE_THRESHOLD_L  = CONF_L # the actual SIDEWAYS override
HMM_PROB_DROP_THRESHOLD = 0.20   # confidence drop -> transition warning
VIX_SPIKE_THRESHOLD     = 0.25   # bar-over-bar VIX jump -> transition warning

REGIME_LABELS = ['H_BULL', 'L_BULL', 'SIDEWAYS', 'L_BEAR', 'H_BEAR']
REGIME_COLORS = {
    'H_BULL':   '#006400',
    'L_BULL':   '#90EE90',
    'SIDEWAYS': '#808080',
    'L_BEAR':   '#FFB6C1',
    'H_BEAR':   '#8B0000',
}

# Feature names follow regdet_v11.py exactly (ret_2h / vol_2h, not ret / vol).
FEATURE_COLS = ['ret_2h', 'mom_1d', 'mom_3d', 'mom_5d',
                'vol_2h', 'vol_expansion', 'vix_chg', 'drawdown', 'dist_ma']

# 5-feature primary subset from the feature-selection analysis (corr clustering
# + PCA + VIF): one representative per information family.
FEATURES_LEAN = ['ret_2h', 'mom_3d', 'vol_2h', 'vol_expansion', 'dist_ma']

BASE_WIN = dict(MOM_1D=1*BARS_PER_DAY, MOM_3D=3*BARS_PER_DAY, MOM_5D=5*BARS_PER_DAY,
                VOL_WIN=10, VOL_FAST=5, VOL_SLOW=20, SWING_WIN=20)

# Per-feature sign/magnitude weights for the subset-agnostic bullishness scorer.
#   raw (DIRECTION_EXCLUDE = ()):
#     score = ret_2h + 0.4*(m1+m3+m5) - vol_2h - vol_expansion - vix_chg - drawdown + dist_ma
#   with FIX 1 (DIRECTION_EXCLUDE = ('vol_2h','vol_expansion')) the two magnitude
#   terms drop out and the score becomes purely directional:
#     score = ret_2h + 0.4*(m1+m3+m5) - vix_chg - drawdown + dist_ma
# The exclusion is applied as a WEIGHT OF ZERO in `direction_weight` below, so it
# stays subset-agnostic: excluding a feature the subset does not contain is a
# no-op, and the HMM's own feature matrix is untouched.
FEATURE_SIGN = {
    'ret_2h': 1.0, 'mom_1d': 1.0, 'mom_3d': 1.0, 'mom_5d': 1.0, 'dist_ma': 1.0,
    'vol_2h': -1.0, 'vol_expansion': -1.0, 'vix_chg': -1.0, 'drawdown': -1.0,
}
FEATURE_MAG = {'mom_1d': 0.4, 'mom_3d': 0.4, 'mom_5d': 0.4}   # else 1.0

# ---------------------------------------------------------------------------
# FIX 5 -- INTENSITY_MODE: the SCALE-FREE H/L intensity gate.
#
# THE FLAW. H_BULL / H_BEAR escalate on a trend-MAGNITUDE gate
#     trend_z = (mom_3d - mu_fit) / sd_fit          |trend_z| >= Z_HI = 0.5
# with mu_fit / sd_fit frozen on the training window. Freezing them is correct
# for CAUSALITY -- but it makes the THRESHOLD meaningless. A constant threshold
# is only interpretable on a scale-free quantity, and trend_z is scale-free only
# if sd_fit happens to equal the CURRENT dispersion of mom_3d. Volatility
# clusters, so it never does: the gate's aggressiveness is governed by the ratio
# sd_fit / sd_now, which is an artefact of where the training cut was placed and
# has no economic meaning.
#
# Real-data evidence (the user's own run, same bars, same config -- ONLY the fit
# window differs):
#     fit 50%:  H_BULL 22.9%  L_BULL 16.2%  SIDEWAYS 36.0%  L_BEAR  8.3%  H_BEAR 16.5%
#     fit 70%:  H_BULL 21.8%  L_BULL 31.0%  SIDEWAYS 10.6%  L_BEAR 17.3%  H_BEAR 19.3%
# SIDEWAYS is 3.4x larger in one than the other. These are effectively two
# different detectors produced by an arbitrary backtest cut.
#
# THE FIX. trend_t = mom_3d / (vol_2h * sqrt(MOM_3D_BARS)) -- the t-statistic of
# the 9-bar move. mom_3d is the 9-bar sum of log returns; vol_2h is the trailing
# per-bar return sd. The ratio is DIMENSIONLESS and CONTEMPORANEOUS, and needs no
# fit-window baseline at all -- the fix REMOVES a fit-window dependence rather
# than relocating it. Both inputs are trailing rolling windows at bar t, so it is
# fully causal.
#
# A REJECTED alternative, measured and discarded: a trailing 250-bar quantile of
# |mom_3d|. It was WORSE than the status quo (H-firing spread 44.7 pp vs 35.8 pp)
# because a trailing window is a LAGGING scale estimate -- when volatility drops
# the window is full of stale high-vol bars and the H-rate collapsed to 11%.
#
# THE THRESHOLD. |trend_t| >= 0.5 would fire on ~68% of bars, so the threshold is
# not hand-picked either: it is derived as a TARGET OCCUPANCY from the FIT WINDOW
# ONLY (causal; computed once from bars <= n_fit, never per bar).
#     H_TARGET_RATE = 0.25  -> enter threshold = the (1 - 0.25) quantile of
#                              |trend_t| over the leading n_fit bars
#     H_EXIT_SLACK  = 0.10  -> exit  threshold = the (1 - 0.35) quantile, i.e. a
#                              LOOSER bar, preserving FIX 3's enter/exit band
# The derived thresholds are PRINTED on every run (Section 7.0).
#
# INTENSITY_MODE = 'frozen_z' reproduces today's behaviour BIT-FOR-BIT and is
# asserted to do so in Section 7H-viii against a frozen verbatim copy of the
# pre-change `label_bars`.
INTENSITY_MODE = 'vol_norm'    # 'frozen_z' = pre-change | 'vol_norm' = adopted
MOM_3D_BARS    = BASE_WIN['MOM_3D']    # 9 -- the horizon mom_3d integrates over
H_TARGET_RATE  = 0.25   # target FIT-WINDOW occupancy of the H gate
H_EXIT_SLACK   = 0.10   # exit threshold sits at (H_TARGET_RATE + this) occupancy

# ---------------------------------------------------------------------------
# FIX 6 -- DIRECTION_MODE: how an HMM state maps to bear / side / bull.
#
# 'rank' (DEFAULT, ADOPTED) -- today's hard rank buckets: sort the N states by
# composite bullishness, bottom 2 bear, middle 1 side, top 2 bull. Rank-based so
# the side bucket is guaranteed non-empty (a sign+deadzone rule can empty it and
# silently make SIDEWAYS unreachable -- that bug was shipped once and reverted).
#
# 'soft' (IMPLEMENTED, SWITCHABLE, *NOT* ADOPTED) -- replace the step function of
# the RANK with a smooth function of the VALUE: two sigmoids on the cross-state
# standardized composite score give each state a (bull, side, bear) weight row,
# and the bar's masses become `probs @ W` instead of a hard column sum.
#
# WHY IT IS NOT ADOPTED, stated exactly. Soft bucketing improves stability AT THE
# SOURCE -- the state->direction map stops flipping wholesale when one state's
# score crosses another's, measured 3.2x more stable -- but it makes the EMITTED
# SIDEWAYS/BEAR occupancy gaps WORSE. The reason is the standardization: with
# only N=5 states the composite scores are standardized by the sd of those same 5
# numbers, so a single outlier state inflates the sd and drags every other
# state's z toward zero, which washes the whole map toward SIDEWAYS by a
# different amount in each fit. Until that standardization is fixed (a robust
# scale, or a scale that does not depend on the state count), 'soft' is NOT
# RECOMMENDED and 'rank' remains the default.
DIRECTION_MODE = 'rank'   # 'rank' = adopted | 'soft' = implemented, not recommended
DIR_TAU   = 0.6    # soft: crossover WIDTH of the bull/bear sigmoids (cross-state sd units)
DIR_C     = 0.5    # soft: crossover CENTRE -- how far from the cross-state mean a
                   #       state must sit before it counts as directional
DIR_SCALE = 'sd'   # soft: cross-state scale, 'sd' or 'mad' (robust). This is the
                   #       knob named in the paragraph above; 'mad' is the obvious
                   #       first thing to try when fixing the standardization.

# ---------------------------------------------------------------------------
# ESCALATION_DURING_HOLD -- what the intensity gate may do while the DIRECTION
# is being held by CONFIRM_BARS.
#
# THE COUPLING. FIX 2 holds the emitted DIRECTION for CONFIRM_BARS bars while a
# candidate flip confirms. The intensity gate is graded FRESH at every bar on a
# SEPARATE clock (its own enter/exit state machine). So on the bars where
# dir_raw != dir_emit -- measured at ~5-11% of bars -- the notebook emits a
# direction that the current evidence no longer supports, while the gate is free
# to escalate that stale direction to H. That is maximum conviction emitted at
# maximum uncertainty.
#
#   'allow'            -- the PRE-CHANGE behaviour. Escalation is independent of
#                         whether the direction is contested. Retained as the
#                         bit-for-bit off switch (asserted in 7H-viii).
#   'block'  (DEFAULT) -- no NEW escalation to H while dir_raw != dir_emit. An
#                         already-running H escalation is still held under the
#                         exit band; only fresh ENTERs are suppressed. A blocked
#                         escalation leaves the gate state at 0, so a LATER bar
#                         cannot "hold" an H run it never entered -- the
#                         suppression propagates forward past the contested bar,
#                         which is what keeps the ENTER-band invariant intact.
#   'demote'           -- as 'block', and additionally force an existing H down to
#                         L while contested. The gate run ENDS, so re-escalation
#                         after the contest resolves must clear the full ENTER
#                         band again.
#
# WHY 'block' IS THE DEFAULT -- and, precisely, what is NOT established.
#
# ESTABLISHED (contested-H prototype):
#   * Contested-H underperforms confirmed-H in 18/18 measured cells (2 fit cuts x
#     3 horizons x {bull, bear, pooled}). Every one of the 18 is negative.
#   * MECHANICAL CORROBORATION, independently verified: contested-H bars carry
#     trend_efficiency 0.365 / 0.445 versus 0.597 / 0.603 for confirmed-H -- i.e.
#     contested-H bars are 1.35-1.64x CHOPPIER. That is exactly the condition this
#     system is supposed to take SMALLER size in, and 'allow' prints MAXIMUM size
#     there.
#   * The cost of switching is negligible and structurally safe: only 6-7 bars
#     change (0.39-0.45%), EVERY change is an H -> L demotion, and NO bar changes
#     DIRECTION. 'block' therefore cannot introduce a new failure mode -- it can
#     only reduce conviction.
#   * Persistence slightly IMPROVES under 'block' (runs 293 -> 291), whereas
#     'demote' fragments runs (-> 311). Hence 'block', not 'demote'.
#
# NOT ESTABLISHED -- read this before quoting the above as a return result:
#   * NO single return comparison clears |t| >= 2 under BOTH the HAC and the n_eff
#     corrections. There are only 16-20 contested-H bars in the sample. That is a
#     POWER limitation, not evidence of no effect -- but it means the 18/18 is a
#     consistent DIRECTION, not a demonstrated return gain.
#   * The justification for defaulting this ON is therefore: consistent sign +
#     the efficiency evidence + the asymmetry of costs (a wrongly-suppressed H
#     costs a little upside; a wrongly-emitted H costs full size into chop).
#     It is NOT "contested-H loses money, significantly". Do not overstate it.
#
# CAUSALITY. Both dir_raw and dir_emit are computable from bars <= t (dir_raw is
# a per-bar argmax of causal masses; dir_emit is confirm_delay's left-to-right
# scan), so the contested mask is causal, and the suppression is applied inside
# the same single left-to-right pass the gate already used. This is PROVED by a
# prefix-truncation probe in Section 7.0 under all three settings, not asserted.
ESCALATION_DURING_HOLD = 'block'   # 'block' (adopted) | 'allow' (pre-change) | 'demote'

# Forward-return horizons used for EVALUATION ONLY (hindsight; never a label input).
FWD_HORIZONS = [3, 9, 15]        # ~1 day, ~3 days, ~5 days of 2h bars

# ---------------------------------------------------------------------------
# The three configs under test: V1.0 (prod baseline) vs Candidate A (lean-cov)
# vs Candidate B (lean-feat). All at N=5, CONF_L=0.5. This head-to-head is about
# MODEL CAPACITY (covariance type / feature subset) -- the labeling scheme below
# is identical for all three, so the comparison isolates capacity.
# ---------------------------------------------------------------------------
CONFIGS = [
    dict(name='V1.0 (prod)',  N=5, cov='full', features=FEATURE_COLS),
    dict(name='A: lean-cov',  N=5, cov='diag', features=FEATURE_COLS),
    dict(name='B: lean-feat', N=5, cov='diag', features=FEATURES_LEAN),
]

# ---------------------------------------------------------------------------
# THE ADOPTED CONFIG -- set EXPLICITLY, not inherited from the head-to-head.
#
# The head-to-head (Section 5) still runs in full and still reports its winner.
# But that ranking is KNOWN UNSTABLE: it is decided on worst-fold Sharpe, a
# single noisy number over 4 folds, and a 3-bar perturbation of the input series
# has already been observed to flip it. Letting the whole evaluation suite follow
# whichever config happened to win means the notebook can silently evaluate a
# different detector on two runs of the same code.
#
# So the evaluated config is PINNED here. Sections 5c and 7 both use it, which is
# also what makes those two overlays comparable: they then differ ONLY in the fit
# window (see the re-role note in 5c / 7A), not in model capacity.
#
# If the head-to-head winner differs from this, that DISAGREEMENT IS REPORTED
# loudly in Sections 5b, 7.0 and 8a rather than silently resolved either way.
ADOPTED_CONFIG_NAME = 'A: lean-cov'    # N=5, cov='diag', all 9 features
assert any(c['name'] == ADOPTED_CONFIG_NAME for c in CONFIGS), \
    f'ADOPTED_CONFIG_NAME {ADOPTED_CONFIG_NAME!r} is not one of the configs under test'

# ---- knob sanity (each of these has bitten this project at least once) ------
assert INTENSITY_MODE in ('frozen_z', 'vol_norm'), 'unknown INTENSITY_MODE'
assert DIRECTION_MODE in ('rank', 'soft'), 'unknown DIRECTION_MODE'
assert ESCALATION_DURING_HOLD in ('allow', 'block', 'demote'), 'unknown ESCALATION_DURING_HOLD'
assert MOM_3D_BARS == BASE_WIN['MOM_3D'] == 9, 'mom_3d is expected to integrate 9 bars'
assert 0.0 < H_TARGET_RATE < 1.0 and H_EXIT_SLACK >= 0.0
assert DIR_TAU > 0.0 and DIR_C >= 0.0 and DIR_SCALE in ('sd', 'mad')
assert not (set(BAR_DIR_FEATURES) & {'vol_2h', 'vol_expansion'}), \
    'a MAGNITUDE feature leaked into the per-bar DIRECTION score'

print(f"Labeling: direction+intensity  CONF_L={CONF_L}  Z_HI={Z_HI}  "
      f"EFF_HI={EFF_HI}  EFF_WIN={EFF_WIN}  TREND_FEATURE={TREND_FEATURE}")
print(f"Fixes   : [1] DIRECTION_EXCLUDE={DIRECTION_EXCLUDE or '() -> old scorer'}  "
      f"[2] CONFIRM_BARS={CONFIRM_BARS}"
      + ('  (=1 -> old behaviour)' if CONFIRM_BARS == 1 else '')
      + f"  [3] gate bands enter(|z|>={Z_HI}, eff>={EFF_HI}) "
        f"exit(|z|<{Z_HI_EXIT}, eff<{EFF_HI_EXIT})"
      + ('  (exit==enter -> old behaviour)'
         if (Z_HI_EXIT == Z_HI and EFF_HI_EXIT == EFF_HI) else ''))
print(f"          [4] BAR_DIR_WEIGHT={BAR_DIR_WEIGHT} (per-bar direction blend; "
      f"tau={BAR_DIR_TAU}, features={list(BAR_DIR_FEATURES)})")
if BAR_DIR_WEIGHT == 0.0:
    print("              FIX 4 IS RETAINED AS A SWITCH BUT SET OFF. Direction is "
          "100% per-STATE (HMM).")
    print("              WHY: it broke the direction-level forward-return ordering, "
          "and the rally")
    print("              benefit it was added for did not reproduce across runs "
          "(+19.7pp, then +1.4pp,")
    print("              then a run where the gain came from fixes 1-2 with fix 4 "
          "already off).")
    print("              NOT deleted: the blend, the 7H-vi sweep and the 7H-vii "
          "overlay all still run,")
    print("              and bar_dir_score is still computed (diagnostic only -- the "
          "blend is an exact")
    print("              arithmetic no-op at w=0.0, asserted bit-for-bit in 7H).")
else:
    print(f"              FIX 4 IS ON. NOTE: w={BAR_DIR_WEIGHT} is NOT the shipped "
          "default (0.0); it broke")
    print("              the direction-level forward-return ordering on real data.")
print(f"Fitting : SEED ENSEMBLE  ENSEMBLE_K={ENSEMBLE_K}  BASE_SEED={BASE_SEED}  "
      f"seeds={[BASE_SEED + i for i in range(ENSEMBLE_K)]}"
      + ('   (K=1 -> old single-fit behaviour)' if ENSEMBLE_K == 1 else ''))
print(f"          [5] INTENSITY_MODE={INTENSITY_MODE!r}"
      + ("  (scale-free t-stat gate; thresholds derived from the FIT WINDOW at "
         f"target occupancy {H_TARGET_RATE:.2f} enter / {H_TARGET_RATE + H_EXIT_SLACK:.2f} exit)"
         if INTENSITY_MODE == 'vol_norm'
         else f"  (pre-change frozen mu/sd z-score vs constants Z_HI={Z_HI}/Z_HI_EXIT={Z_HI_EXIT})"))
print(f"          [6] DIRECTION_MODE={DIRECTION_MODE!r}"
      + ("  (hard rank buckets -- adopted)" if DIRECTION_MODE == 'rank'
         else f"  (SOFT weights tau={DIR_TAU} c={DIR_C} scale={DIR_SCALE!r}"
              "  -- NOT RECOMMENDED, see the note in the config cell)"))
print(f"          [7] ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}"
      + ("  (=allow -> PRE-CHANGE behaviour: the gate may escalate a held/contested direction)"
         if ESCALATION_DURING_HOLD == 'allow'
         else "  (no NEW H escalation while dir_raw != dir_emit"
              + ("; existing H also demoted)" if ESCALATION_DURING_HOLD == 'demote' else ")")))
print(f"Harness : HMM_ITER={HMM_ITER}  N_FOLDS={N_FOLDS}  MIN_TRAIN_FRAC={MIN_TRAIN_FRAC}  "
      f"configs={[c['name'] for c in CONFIGS]}")
print(f"Adopted : evaluated config is PINNED to {ADOPTED_CONFIG_NAME!r} (Sections 5c and 7); "
      f"the head-to-head still runs and its winner is reported and compared in 5b / 7.0 / 8a")
print(f"Runtime : N_JOBS={N_JOBS} (ensemble fits" + ('  serial' if N_JOBS == 1 else ' in parallel')
      + f")  RUN_SEED_STABILITY={RUN_SEED_STABILITY}"
      + ('  (5d diagnostic SKIPPED -- see 5d)' if not RUN_SEED_STABILITY else '')
      + '   [speed only; neither knob can change a result]')

In [ ]:
# ==========================================================================
# DATA LOADER -- INLINED VERBATIM (master code cell 5).
# yfinance 60m -> 2h resample, with a synthetic fallback. Unmodified.
# ==========================================================================
def load_2h():
    """yfinance 60m->2h, else synthetic. Returns (nifty, vix, is_synth)."""
    try:
        import yfinance as yf

        def h2(tk):
            raw = yf.download(tk, interval='60m', period='730d',
                              auto_adjust=True, progress=False)
            if raw is None or len(raw) == 0:
                return None
            c = raw['Close'].squeeze().dropna()
            idx = pd.to_datetime(c.index)
            c.index = idx.tz_localize(None) if idx.tz is not None else idx   # naive index
            return c.resample('2h').last().dropna()

        nifty = h2('^NSEI')
        if nifty is not None and len(nifty) > 200:
            vix = h2('^INDIAVIX')
            vix = (vix.reindex(nifty.index).ffill().bfill()
                   if vix is not None and len(vix) else pd.Series(15.0, index=nifty.index))
            nifty.name, vix.name = 'nifty', 'vix'
            print(f'yfinance 60m->2h: {len(nifty)} bars')
            return nifty, vix, False
        raise ValueError('too few bars')
    except Exception as e:
        print(f'yfinance unavailable ({e}); falling back to synthetic 2h data.')
        return _synth()


def _synth():
    """Regime-blocked GBM + OU VIX at 2h cadence (offline validation only)."""
    rng = np.random.default_rng(42)
    days = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=520)
    idx = pd.DatetimeIndex([d + pd.Timedelta(hours=h) for d in days for h in (10, 12, 14)])
    REG = [(0.00, 0.15,  0.0004, 0.0035, 14.0, 0.6),
           (0.15, 0.30, -0.0006, 0.0060, 22.0, 1.1),
           (0.30, 0.45,  0.0001, 0.0030, 16.0, 0.7),
           (0.45, 0.55, -0.0030, 0.0110, 45.0, 2.5),
           (0.55, 0.75,  0.0006, 0.0050, 20.0, 1.0),
           (0.75, 0.90,  0.0005, 0.0032, 13.5, 0.6),
           (0.90, 1.00, -0.0002, 0.0045, 18.0, 0.9)]
    n, lvl, pv, nv, vv = len(idx), 18000.0, 15.0, [], []
    for i in range(n):
        f = i / n
        mu, sg, vm, vf = 0.0002, 0.0040, 16.0, 0.8
        for fs, fe, m, s, v, q in REG:
            if fs <= f < fe:
                mu, sg, vm, vf = m, s, v, q
                break
        lvl *= np.exp(rng.normal(mu, sg))
        pv = max(pv + 0.05 * (vm - pv) + rng.normal(0, vm * 0.08 * vf), 8.0)
        nv.append(lvl); vv.append(pv)
    return (pd.Series(nv, index=idx, name='nifty'),
            pd.Series(vv, index=idx, name='vix'), True)


nifty, vix, TAC_SYNTH = load_2h()
if TAC_SYNTH:
    print(f"\n*** TAC_SYNTH = True  ->  SYNTHETIC data ({len(nifty)} bars). "
          f"Results below are ILLUSTRATIVE ONLY. ***\n")
else:
    print(f"\n*** TAC_SYNTH = False  ->  REAL yfinance data ({len(nifty)} bars). "
          f"Results below are decision-grade. ***\n")
print(f"Span: {nifty.index[0]}  ->  {nifty.index[-1]}")

In [ ]:
# ==========================================================================
# FEATURE + LABELING ENGINE -- INLINED VERBATIM (master code cell 7).
# build_features / fit_hmm_ensemble / confirm_delay / label_bars / n_params.
# ==========================================================================
def build_features(nifty, vix, scale=LOOKBACK_SCALE):
    """The 9 causal swing features (engine column names)."""
    w = {k: max(2, int(round(v * scale))) for k, v in BASE_WIN.items()}
    r = np.log(nifty / nifty.shift(1))
    df = pd.DataFrame(index=nifty.index)
    df['ret_2h'] = r
    df['mom_1d'] = r.rolling(w['MOM_1D']).sum()
    df['mom_3d'] = r.rolling(w['MOM_3D']).sum()
    df['mom_5d'] = r.rolling(w['MOM_5D']).sum()
    df['vol_2h'] = r.rolling(w['VOL_WIN']).std()
    df['vol_expansion'] = r.rolling(w['VOL_FAST']).std() / r.rolling(w['VOL_SLOW']).std()
    df['vix_chg'] = (vix - vix.shift(1)) / vix.shift(1)
    sh = nifty.rolling(w['SWING_WIN']).max()
    df['drawdown'] = (sh - nifty) / sh
    ma = nifty.rolling(w['SWING_WIN']).mean()
    df['dist_ma'] = (nifty - ma) / ma
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    return df


def direction_weight(feat, exclude=None):
    """Weight this feature contributes to the composite BULLISHNESS score.

    FIX 1: features listed in `exclude` (default DIRECTION_EXCLUDE) get weight
    0.0 -- they stop voting on DIRECTION while remaining full HMM inputs.
    vol_2h and vol_expansion are magnitude measures, not signed ones; scoring a
    high-volatility RALLY as bearish was the bug this removes.

    Subset-agnostic: excluding a feature that is not in the active subset is a
    no-op, and `exclude=()` reproduces the original weights exactly.
    """
    exclude = DIRECTION_EXCLUDE if exclude is None else exclude
    if feat in exclude:
        return 0.0
    return FEATURE_SIGN[feat] * FEATURE_MAG.get(feat, 1.0)


def composite_subset(means, feature_subset, exclude=None):
    """Bullishness score per state; works with ANY feature subset."""
    score = np.zeros(means.shape[0])
    for j, feat in enumerate(feature_subset):
        score += direction_weight(feat, exclude) * means[:, j]
    # A subset whose every feature is excluded would make all states score 0 and
    # the direction ranking arbitrary -- catch that rather than emit noise.
    assert any(direction_weight(f, exclude) != 0.0 for f in feature_subset), \
        'DIRECTION_EXCLUDE removed every feature from the direction score'
    return score


def n_params(N, F, covariance_type):
    """GaussianHMM free parameters: means + covariances + transitions + startprob."""
    cov = N * F * (F + 1) / 2 if covariance_type == 'full' else N * F
    return N * F + cov + N * (N - 1) + (N - 1)


def trend_efficiency(close, win=EFF_WIN):
    """Causal Kaufman efficiency ratio over `win` bars:

        |close[t] - close[t-win]| / sum(|close.diff()|)

    i.e. net displacement / total path walked. ~1.0 in a straight-line trend,
    ~0.0 when price keeps doubling back inside a range -- the distinction raw
    momentum magnitude cannot make. Uses only bars <= t.
    """
    net = (close - close.shift(win)).abs()
    path = close.diff().abs().rolling(win).sum()
    return (net / path.replace(0, np.nan)).fillna(0.0).clip(0.0, 1.0)


def gate_band(trend_raw, feat, n_fit, mode=None, thr_enter=None, thr_exit=None,
              target_rate=None, exit_slack=None, hysteresis=None):
    """ONE band contract for BOTH intensity modes.

    Returns `(trend, thr_enter, thr_exit)`: the trend-MAGNITUDE series and the
    (enter, exit) thresholds it is graded against.

    THIS FUNCTION EXISTS TO RESOLVE A REAL COUPLING. In the prototype, the gate
    BAND (FIX 3's Z_HI_EXIT / EFF_HI_EXIT) and the INTENSITY MODE (FIX 5) were
    mutually exclusive: the vol_norm branch asserted `z_exit is None`, because
    z_exit was a threshold expressed in frozen-z units and vol_norm derives its
    thresholds by occupancy instead. That is a units problem, not a logic
    problem, and it is fixed here by making the BAND -- not the threshold
    constants -- the shared abstraction. Both modes now:

      * produce a trend series in their OWN units,
      * carry a DEFAULT (enter, exit) band in those same units,
      * accept an OCCUPANCY-derived band (`target_rate` / `exit_slack`) computed
        on the FIT WINDOW of that same series,
      * accept EXPLICIT overrides (`thr_enter` / `thr_exit`) in those same units,
      * accept `hysteresis=False`, a MODE-INDEPENDENT way to say "no band"
        (exit == enter), which is what FIX-3-off means in either mode.

    So the mode and the band are now independent knobs, and nothing is papered
    over with an assert.

    mode='frozen_z' : trend = (mom_3d - mu_fit)/sd_fit.
                      Default band = (Z_HI, Z_HI_EXIT), the pre-change constants.
    mode='vol_norm' : trend = mom_3d / (vol_2h * sqrt(MOM_3D_BARS)) -- the
                      t-statistic of the 9-bar move: dimensionless,
                      contemporaneous, needing NO fit-window baseline.
                      Default band = occupancy quantiles at H_TARGET_RATE /
                      H_TARGET_RATE + H_EXIT_SLACK.

    PRECEDENCE (most specific wins): explicit thr_* > occupancy (target_rate /
    exit_slack, honoured under EITHER mode) > the mode default. `hysteresis=False`
    is applied last and collapses exit onto enter whatever produced them.

    CAUSALITY. Under frozen_z, mu/sd come from the leading n_fit bars. Under
    vol_norm the SERIES needs no fit window at all (mom_3d and vol_2h are both
    trailing rolling windows at bar t) and the THRESHOLDS are quantiles over the
    leading n_fit bars only -- computed ONCE, never per bar, never over bars the
    model has not seen. Either way a prefix truncation at any t >= n_fit
    reproduces both the series value at t and the thresholds exactly. Section 7.0
    proves this by truncation rather than asserting it here.
    """
    mode = INTENSITY_MODE if mode is None else mode
    assert mode in ('frozen_z', 'vol_norm'), f'unknown INTENSITY_MODE {mode!r}'
    tr = np.asarray(trend_raw, dtype=float)
    n_fit = int(n_fit)
    assert 2 <= n_fit <= len(tr), 'fit window out of range for the intensity gate'

    if mode == 'frozen_z':
        mu = float(np.mean(tr[:n_fit]))
        sd = float(np.std(tr[:n_fit]))
        trend = (tr - mu) / (sd if sd > 0 else 1.0)
        ok = np.isfinite(trend)
        d_enter, d_exit = float(Z_HI), float(Z_HI_EXIT)
    else:
        assert feat is not None and 'vol_2h' in getattr(feat, 'columns', []), \
            "vol_norm needs the raw feature frame (for vol_2h)"
        vol = np.asarray(feat['vol_2h'].values, dtype=float)
        assert len(vol) == len(tr), 'vol_2h must be aligned 1:1 with the trend feature'
        # DENOMINATOR GUARD: vol_2h is NaN through warm-up and 0.0 on a dead-flat
        # stretch. Those bars get trend = 0.0 -- the neutral value, which fires no
        # gate and can only SUPPRESS an escalation, never invent one.
        den = vol * np.sqrt(MOM_3D_BARS)
        ok = np.isfinite(den) & (den > 0.0) & np.isfinite(tr)
        trend = np.zeros(len(tr), dtype=float)
        np.divide(tr, den, out=trend, where=ok)
        trend[~np.isfinite(trend)] = 0.0
        d_enter = d_exit = None                 # derived by occupancy below

    # ---- occupancy-derived band (either mode) -----------------------------
    if d_enter is None or target_rate is not None or exit_slack is not None:
        tgt = H_TARGET_RATE if target_rate is None else float(target_rate)
        slk = H_EXIT_SLACK if exit_slack is None else float(exit_slack)
        assert 0.0 < tgt < 1.0 and slk >= 0.0, 'target occupancy out of range'
        a = np.abs(trend[:n_fit])[ok[:n_fit]]   # guarded bars excluded from the quantile
        assert a.size >= 20, 'too few usable fit-window bars to derive a threshold'
        d_enter = float(np.quantile(a, 1.0 - tgt))
        d_exit = float(np.quantile(a, 1.0 - min(tgt + slk, 0.999)))

    en = float(d_enter) if thr_enter is None else float(thr_enter)
    ex = float(d_exit) if thr_exit is None else float(thr_exit)
    if hysteresis is False:
        ex = en                                 # FIX-3-OFF, stated mode-independently
    ex = min(ex, en)                            # the exit band must be the looser one
    return trend, en, ex


def intensity_state(z, eff, z_hi=None, eff_hi=None, z_exit=None, eff_exit=None,
                    block_enter=None, force_exit=None):
    """FIX 3 -- gate hysteresis. Returns a signed intensity array:

        +1  bar is escalated H on the BULL side
        -1  bar is escalated H on the BEAR side
         0  bar stays L

    Enter (0 -> +/-1): |z| >= z_hi AND eff >= eff_hi          (unchanged gates)
    Hold  (stay +/-1): |z| >= z_exit AND eff >= eff_exit AND sign(z) unchanged
    Exit  (-> 0):      |z| <  z_exit OR  eff <  eff_exit OR  sign(z) flipped

    A sign flip forces a fresh ENTER test rather than silently relabelling a held
    bull escalation as a bear one.

    CAUSAL: a single left-to-right scan whose state at bar t depends only on
    bars <= t, so truncating the input after t cannot change out[t].

    z_exit == z_hi and eff_exit == eff_hi reduce this EXACTLY to the old
    memoryless `(|z| >= z_hi) & (eff >= eff_hi)` test.

    ESCALATION_DURING_HOLD support -- two OPTIONAL per-bar boolean masks, applied
    INSIDE this same single pass so the state machine stays consistent (a
    suppressed escalation must not be silently "held" on a later bar as though it
    had happened):

      block_enter[t] : the ENTER test is skipped at bar t. An already-running
                       escalation is unaffected and is still held under the exit
                       band. This is 'block'.
      force_exit[t]  : the state is additionally forced to 0 at bar t, ending the
                       run. Re-escalation later must clear the full ENTER band
                       again. This is the extra half of 'demote'.

    Both masks default to all-False, in which case every branch below is exactly
    the pre-change scan -- and Section 7H-viii asserts that bit-for-bit rather
    than trusting this paragraph. Neither mask can CREATE an escalation; both can
    only suppress one, so no chop-filter invariant can be weakened by them.

    CAUSALITY IS PRESERVED BY CONSTRUCTION: mask[t] is consumed at step t of a
    left-to-right scan, so out[t] still depends only on (z, eff, masks)[0..t].
    """
    z_hi = Z_HI if z_hi is None else z_hi
    eff_hi = EFF_HI if eff_hi is None else eff_hi
    z_exit = Z_HI_EXIT if z_exit is None else z_exit
    eff_exit = EFF_HI_EXIT if eff_exit is None else eff_exit
    z = np.asarray(z, dtype=float)
    eff = np.asarray(eff, dtype=float)
    n = len(z)
    _blk = (np.zeros(n, dtype=bool) if block_enter is None
            else np.asarray(block_enter, dtype=bool))
    _fex = (np.zeros(n, dtype=bool) if force_exit is None
            else np.asarray(force_exit, dtype=bool))
    assert len(_blk) == n and len(_fex) == n, 'gate suppression masks must align 1:1 with z'
    out = np.zeros(n, dtype=int)
    state = 0
    for t in range(n):
        s = 1 if z[t] > 0 else (-1 if z[t] < 0 else 0)
        if state != 0 and s == state:
            if abs(z[t]) < z_exit or eff[t] < eff_exit:
                state = 0                       # de-escalate on the EXIT band
        else:
            state = 0                           # no state, or the sign flipped
        if _fex[t]:
            state = 0                           # 'demote': drop a held H to L
        if (state == 0 and not _blk[t]
                and abs(z[t]) >= z_hi and eff[t] >= eff_hi):
            state = s                           # escalate on the ENTER band
        out[t] = state
    return out


def hold_masks(dir_raw, dir_emit, policy=None):
    """(block_enter, force_exit) for ESCALATION_DURING_HOLD.

    A bar is CONTESTED when the emitted direction is not the direction the
    current bar's own evidence votes for -- i.e. `dir_raw[t] != dir_emit[t]`,
    which is exactly the set of bars CONFIRM_BARS is holding through. Both inputs
    are computable from bars <= t, so the mask is causal.

    'allow'  -> (all False, all False)  == unchanged behaviour, bit-for-bit.
    'block'  -> (contested, all False)  == no NEW escalation while contested.
    'demote' -> (contested, contested)  == also drop an existing H to L.
    """
    policy = ESCALATION_DURING_HOLD if policy is None else policy
    assert policy in ('allow', 'block', 'demote'), f'unknown ESCALATION_DURING_HOLD {policy!r}'
    n = len(dir_raw)
    contested = np.asarray(dir_raw) != np.asarray(dir_emit)
    if policy == 'allow':
        z = np.zeros(n, dtype=bool)
        return z, z.copy(), contested
    if policy == 'block':
        return contested, np.zeros(n, dtype=bool), contested
    return contested, contested.copy(), contested


def confirm_delay(raw, confirm_bars=None):
    """FIX 2 -- causal label hysteresis (a CONFIRMATION DELAY, not a smoother).

    A candidate value must be observed on `confirm_bars` CONSECUTIVE bars before
    the emitted series is allowed to flip to it; until then the previous emitted
    value is held.

    WHY THIS IS CAUSAL, stated precisely: out[t] is a function of raw[0..t] only.
    The scan never looks at raw[t+1..]. Contrast with the look-ahead smoother
    this project previously removed, which decided whether to erase a run by
    inspecting that run's full REALIZED length -- i.e. it needed bars after t to
    decide bar t. This does the opposite: it PAYS a delay rather than borrowing
    the future. A flip that turns out to be a 1-bar blip is simply never emitted;
    a flip that persists is emitted `confirm_bars - 1` bars late.

    confirm_bars = 1 reproduces the input exactly (out is raw).
    """
    confirm_bars = CONFIRM_BARS if confirm_bars is None else int(confirm_bars)
    assert confirm_bars >= 1, 'CONFIRM_BARS must be >= 1'
    raw = np.asarray(raw)
    n = len(raw)
    out = np.empty(n, dtype=raw.dtype)
    if n == 0:
        return out
    out[0] = raw[0]                 # bar 0 has no prior emitted label to hold
    cand, run = raw[0], 0
    for t in range(1, n):
        if raw[t] == out[t - 1]:
            out[t] = raw[t]         # agrees with what is already emitted
            cand, run = raw[t], 0
        else:
            if raw[t] == cand:
                run += 1
            else:
                cand, run = raw[t], 1
            if run >= confirm_bars:
                out[t] = cand       # candidate confirmed -> flip
                run = 0
            else:
                out[t] = out[t - 1]  # not yet confirmed -> HOLD the old label
    return out


def direction_buckets(means, feature_subset, exclude=None):
    """Bucket HMM states into bear(-1) / side(0) / bull(+1) by RANK of composite
    bullishness. Rank-based (not a sign+deadzone threshold) so the side bucket is
    guaranteed non-empty; a deadzone can degenerate to an empty side bucket, which
    silently makes SIDEWAYS unreachable. For N=5 this is: bottom 2 bear, middle 1
    side, top 2 bull.

    `exclude` is threaded to the scorer (FIX 1); None uses DIRECTION_EXCLUDE.
    """
    scores = composite_subset(means, feature_subset, exclude)
    n_st = len(scores)
    order = np.argsort(scores)                 # ascending bullishness
    n_side = max(1, round(n_st / 5))
    n_bear = (n_st - n_side) // 2
    direction = np.empty(n_st, dtype=int)
    direction[order[:n_bear]] = -1
    direction[order[n_bear:n_bear + n_side]] = 0
    direction[order[n_bear + n_side:]] = 1
    # INVARIANT: every direction bucket must be reachable, else whole labels vanish.
    assert (direction == 1).sum() >= 1, 'bull bucket empty -> H_BULL/L_BULL unreachable'
    assert (direction == -1).sum() >= 1, 'bear bucket empty -> H_BEAR/L_BEAR unreachable'
    assert (direction == 0).sum() >= 1, 'side bucket empty -> SIDEWAYS unreachable'
    return direction


def bar_direction_score(dir_feats, n_fit, features=None):
    """FIX 4 -- CAUSAL PER-BAR directional z-score.

    dir_feats : DataFrame of RAW (unscaled) features aligned to the labeled bars,
                one row per bar, in bar order. Only the signed directional columns
                are read.
    n_fit     : number of LEADING bars that constitute the fit window. The mu/sd
                baseline is computed from those rows ONLY -- exactly the way
                trend_z's baseline is frozen -- so no bar > t and no bar the model
                has not seen can influence bar t's score.

    Construction:
      1. keep only BAR_DIR_FEATURES that are present (ret_2h, mom_1d, mom_3d,
         mom_5d, dist_ma -- all SIGNED directional measures);
      2. z-score each against its FIT-WINDOW mu/sd;
      3. weighted mean with the existing FEATURE_SIGN * FEATURE_MAG weights,
         normalised by sum|w| so the composite stays on a z-like scale;
      4. re-standardize the composite against ITS fit-window mu/sd, so the output
         is a unit-variance z on the fit window and BAR_DIR_TAU is interpretable.

    vol_2h / vol_expansion are structurally barred: they measure how BIG a move
    is, not which way it points, and letting a magnitude term vote on direction
    is the category error this whole fix exists to undo. Asserted below.

    Causality: every input column is a backward-looking rolling statistic, and
    the baseline uses leading rows only, so score[t] depends on bars <= t alone.
    Recomputing from the prefix dir_feats.iloc[:t+1] reproduces score[t] exactly
    (probed in 7.0).
    """
    features = BAR_DIR_FEATURES if features is None else tuple(features)
    assert not (set(features) & {'vol_2h', 'vol_expansion'}), \
        'per-bar DIRECTION score must not contain magnitude features'
    cols = [f for f in features if f in dir_feats.columns]
    assert cols, 'no directional features available for the per-bar direction score'
    A = np.asarray(dir_feats[cols].values, dtype=float)
    assert n_fit >= 2 and n_fit <= len(A), 'fit window out of range for the per-bar score'
    mu = A[:n_fit].mean(axis=0)
    sd = A[:n_fit].std(axis=0)
    Z = (A - mu) / np.where(sd > 0, sd, 1.0)
    # exclude=() deliberately: DIRECTION_EXCLUDE is FIX 1's state-level knob and
    # must not be able to mute a feature that is already guaranteed directional.
    w = np.array([direction_weight(f, exclude=()) for f in cols], dtype=float)
    assert np.abs(w).sum() > 0, 'per-bar direction weights are all zero'
    # (Z * w).sum(axis=1), NOT Z @ w. The two are algebraically identical and the
    # matmul is the obvious way to write it -- but `DataFrame.values` hands back an
    # F-ORDERED array, and BLAS gemv on an F-ordered operand picks its blocking
    # from the ROW COUNT, so the last-bar result changes in the last ulp depending
    # on how many bars follow it. That is a ~1e-16 difference with no economic
    # meaning, but it makes the truncation probe in 7.0 fail its bit-exactness
    # test, and a causality probe that has to be run at a tolerance is a weaker
    # probe. The row-wise form sums 5 terms per row independently of the array
    # length, so score[t] is BIT-identical whether or not bars > t exist -- and
    # 7.0 can therefore assert exact equality rather than np.isclose.
    s = (Z * w).sum(axis=1) / np.abs(w).sum()
    s_mu = float(s[:n_fit].mean())
    s_sd = float(s[:n_fit].std())
    return (s - s_mu) / (s_sd if s_sd > 0 else 1.0)


def bar_direction_masses(score, tau=None):
    """Map the per-bar directional z to bull / side / bear masses that sum to 1.

    Softmax over the three logits (+s/tau, 0, -s/tau): bull dominates for s >> 0,
    bear for s << 0, and SIDE is the plurality only near s ~ 0 -- which is the
    right shape, because "no clear direction at this bar" is a real answer and
    must remain reachable. Computed in a shift-stabilised form so large |s| does
    not overflow.

    Returns (bull, side, bear), each an array over bars, summing to 1 per bar.
    """
    tau = BAR_DIR_TAU if tau is None else float(tau)
    e = np.asarray(score, dtype=float) / max(tau, 1e-12)
    a = np.abs(e)                                   # = max(e, 0, -e), the shift
    eb, es, er = np.exp(e - a), np.exp(-a), np.exp(-e - a)
    tot = eb + es + er
    bull, side, bear = eb / tot, es / tot, er / tot
    assert np.allclose(bull + side + bear, 1.0, atol=1e-9), \
        'per-bar direction masses must partition to 1'
    return bull, side, bear


def _filtered_posteriors(model, X):
    """
    CAUSAL (filtered) state posteriors: P(state_t | observations_1..t).

    Why this exists instead of model.predict_proba():
      hmmlearn's predict_proba runs forward-BACKWARD, so the posterior it
      reports for bar t is smoothed using the whole sequence — including bars
      AFTER t. That is legitimate for offline sequence analysis but is
      look-ahead for a trading regime label: on 2h Nifty data it changes the
      winning state on ~6% of bars versus what was actually knowable at the
      time. predict() (Viterbi) has the same whole-sequence property.

      This is the forward (alpha) recursion only, so each bar's posterior is
      conditioned solely on information available at that bar — exactly what a
      live engine would have. The last bar of a forward-backward pass happens
      to equal the filtered value (no future exists yet), which is why LIVE
      calls were always correct; it is the HISTORICAL labels, and therefore
      every backtest built on them, that needed this fix.

    Computed by the SCALED forward algorithm: alpha is held in LINEAR space and
    renormalised to sum 1 at every step. See `_filtered_posteriors_logspace`
    below for the original log-space/logsumexp formulation, which this is checked
    against bar-by-bar in Section 7.0.

    WHY THE SCALED FORM IS THE SAME ANSWER. The quantity wanted here is the
    NORMALISED filtered posterior at each t, which is scale-free in alpha: for
    any c_t > 0, normalising c_t * alpha_t gives the identical row. So the
    per-step renormalisation -- which is what the log-space version was already
    doing, just via logsumexp -- is not an approximation, it IS the answer. The
    emission frame is likewise exponentiated after subtracting its per-row max,
    another positive per-row constant that cancels in the same normalisation.
    Nothing accumulates, so nothing underflows: alpha sums to 1 after every bar.

    WHY IT IS FASTER. The recursion over t is inherently sequential and is NOT
    vectorised across t (doing so would be wrong). What changes is the cost of
    each step: two scipy.special.logsumexp calls plus an (N,N) broadcast add and
    an exp become one length-N matrix-vector product, one multiply and one
    divide. logsumexp is a Python-level function doing max/subtract/exp/sum/log
    over an (N,N) array per bar; on ~2.9k bars per model per labeling call, and
    hundreds of such calls, that dominates the labeling cost.

    A guard falls back to the log-space implementation if the linear recursion
    ever produces a non-finite or non-positive normaliser, so the fast path can
    never silently return a degraded answer.
    """
    log_frame = model._compute_log_likelihood(X)          # (T, n_states)
    tiny      = np.finfo(float).tiny
    start     = model.startprob_ + tiny
    trans     = model.transmat_ + tiny

    T, N = log_frame.shape
    out  = np.empty((T, N), dtype=float)

    # exp of the emission log-likelihoods, per-bar max removed. The removed
    # factor is a positive per-row constant and cancels in the normalisation.
    frame = np.exp(log_frame - log_frame.max(axis=1, keepdims=True))

    alpha = start * frame[0]
    s = alpha.sum()
    if not (s > 0.0 and np.isfinite(s)):
        return _filtered_posteriors_logspace(model, X)
    alpha = alpha / s
    out[0] = alpha

    for t in range(1, T):
        # predict step (transition) then update step (emission at bar t)
        alpha = (alpha @ trans) * frame[t]
        s = alpha.sum()
        if not (s > 0.0 and np.isfinite(s)):
            return _filtered_posteriors_logspace(model, X)
        alpha = alpha / s                                 # renormalise each step
        out[t] = alpha

    return out


def _filtered_posteriors_logspace(model, X):
    """The ORIGINAL log-space / logsumexp forward recursion.

    Kept verbatim as (a) the reference implementation that
    `_filtered_posteriors` is asserted equal to in Section 7.0, and (b) the
    fallback if the scaled recursion ever hits a degenerate normaliser. Slower,
    but identical in what it computes.
    """
    from scipy.special import logsumexp

    log_frame = model._compute_log_likelihood(X)          # (T, n_states)
    tiny      = np.finfo(float).tiny
    log_start = np.log(model.startprob_ + tiny)
    log_trans = np.log(model.transmat_ + tiny)

    T, N = log_frame.shape
    out  = np.empty((T, N), dtype=float)

    log_alpha = log_start + log_frame[0]
    log_alpha -= logsumexp(log_alpha)
    out[0] = np.exp(log_alpha)

    for t in range(1, T):
        # predict step (transition) then update step (emission at bar t)
        log_alpha = logsumexp(log_alpha[:, None] + log_trans, axis=0) + log_frame[t]
        log_alpha -= logsumexp(log_alpha)                 # renormalise each step
        out[t] = np.exp(log_alpha)

    return out


def ensemble_seeds(K=None, base_seed=None):
    """Deterministic seed list for the ensemble: base_seed + 0..K-1."""
    K = ENSEMBLE_K if K is None else int(K)
    base_seed = BASE_SEED if base_seed is None else int(base_seed)
    return [base_seed + i for i in range(K)]


def _fit_one_hmm(X_train, n_components, covariance_type, n_iter, sd):
    """ONE ensemble member. Top-level (not a closure) so joblib can pickle it.

    Fully determined by its arguments: the seed is explicit, so this touches no
    global RNG state and is identical whether it runs in this process or a
    worker. This is the only place a GaussianHMM is constructed and fit.
    """
    m = hmm.GaussianHMM(n_components=n_components, covariance_type=covariance_type,
                        n_iter=n_iter, random_state=sd,
                        init_params='stmc', params='stmc')
    m.fit(X_train)
    return m


def fit_hmm_ensemble(X_train, n_components, covariance_type,
                     K=None, base_seed=None, n_iter=None):
    """Fit K GaussianHMMs on the SAME training slice with K different seeds.

    This is the identifiability fix. EM is a local optimizer; one seed gives one
    arbitrary local optimum. K seeds give K samples of the optimum set, whose
    direction-bucket masses are averaged in `ensemble_direction_masses` below.

    Every model sees EXACTLY the same rows (`X_train`), which must already be
    the causal leading slice -- this function does no slicing of its own, so it
    cannot introduce look-ahead.

    The K fits are INDEPENDENT and each is fully determined by its own
    random_state, so with N_JOBS != 1 they are dispatched concurrently via
    joblib. That is a pure scheduling change: no fit can observe another, and
    none of them consumes global RNG state (each gets an explicit seed). N_JOBS=1
    takes the plain serial loop. Section 7.0 asserts the two paths return
    BIT-IDENTICAL models.

    Returns (models, all_converged).
    """
    n_iter = HMM_ITER if n_iter is None else int(n_iter)
    seeds = ensemble_seeds(K, base_seed)

    if N_JOBS == 1 or len(seeds) == 1:
        models = [_fit_one_hmm(X_train, n_components, covariance_type, n_iter, sd)
                  for sd in seeds]
    else:
        from joblib import Parallel, delayed
        models = Parallel(n_jobs=N_JOBS, backend='loky')(
            delayed(_fit_one_hmm)(X_train, n_components, covariance_type, n_iter, sd)
            for sd in seeds)

    all_conv = all(bool(m.monitor_.converged) for m in models)
    return list(models), all_conv


def ensemble_direction_masses(models, Xs, feature_subset, exclude=None):
    """Average the direction-bucket probability masses across an ensemble.

    For each fitted model:
      1. CAUSAL filtered posteriors via `_filtered_posteriors` (forward-only
         alpha recursion). Never predict/predict_proba over the whole sequence --
         those are forward-BACKWARD/Viterbi and smooth bar t with bars after t.
      2. `direction_buckets` maps that model's states to bear/side/bull.
      3. The per-state posterior collapses to 3 columns: bull / side / bear.

    Those 3-column arrays are then averaged across models. This is only valid
    because direction masses are PERMUTATION-INVARIANT: model A's "state 3" and
    model B's "state 1" are unrelated integers, but "the probability mass sitting
    in bullish states" means the same thing in both. Averaging raw per-state
    posteriors would be meaningless.

    Causality is preserved exactly: the average of K quantities each of which
    depends only on bars <= t depends only on bars <= t.

    Returns (bull_mass, side_mass, bear_mass, probs_model0), where probs_model0
    is the first model's per-state filtered posterior (used only for the
    `hmm_state_int` reporting column, which has no ensemble analogue).
    """
    models = list(models)
    assert len(models) >= 1, 'ensemble must contain at least one model'
    acc = np.zeros((len(Xs), 3), dtype=float)      # columns: bull, side, bear
    probs0 = None
    for i, m in enumerate(models):
        direction = direction_buckets(m.means_, feature_subset, exclude)
        probs = _filtered_posteriors(m, Xs)        # CAUSAL, forward-only
        if i == 0:
            probs0 = probs
        acc[:, 0] += probs[:, direction == 1].sum(axis=1)
        acc[:, 1] += probs[:, direction == 0].sum(axis=1)
        acc[:, 2] += probs[:, direction == -1].sum(axis=1)
    acc /= len(models)
    # INVARIANT: an average of rows that each sum to 1 must itself sum to 1.
    assert np.allclose(acc.sum(axis=1), 1.0, atol=1e-9), \
        'ensembled direction masses must partition to 1'
    return acc[:, 0], acc[:, 1], acc[:, 2], probs0


# ---------------------------------------------------------------------------
# FIX 6 -- DIRECTION_MODE = 'soft'. IMPLEMENTED AND SWITCHABLE, NOT ADOPTED.
#
# READ THIS BEFORE TURNING IT ON. Soft bucketing improves stability AT THE SOURCE
# (the state -> direction map stops flipping wholesale when one state's composite
# score crosses another's -- measured 3.2x more stable) but it makes the EMITTED
# SIDEWAYS / BEAR occupancy gaps WORSE. The mechanism is the standardization
# below: with only N = 5 states the composite scores are standardized by the sd
# of those same 5 numbers, so a single outlier state inflates the sd and drags
# every other state's z toward zero, washing the map toward SIDEWAYS by a
# different amount in each fit. IT IS NOT RECOMMENDED UNTIL THE STANDARDIZATION
# IS FIXED (DIR_SCALE='mad' is the obvious first thing to try). DIRECTION_MODE
# stays 'rank'.
# ---------------------------------------------------------------------------
def _sigmoid(x):
    """Overflow-free logistic. exp is evaluated only on the non-positive side."""
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    p, n = x >= 0, x < 0
    out[p] = 1.0 / (1.0 + np.exp(-x[p]))
    e = np.exp(x[n])
    out[n] = e / (1.0 + e)
    return out


def soft_direction_weights(means, feature_subset, exclude=None, tau=None, c=None,
                           require_reachable=True, scale=None):
    """SOFT replacement for `direction_buckets`. Returns (W, z, scores).

    W is an (N, 3) row-stochastic matrix with columns [bull, side, bear]. The
    scorer is `composite_subset` -- the notebook's own, unchanged -- so FIX 1
    (`DIRECTION_EXCLUDE`) is threaded through untouched and 'soft' reads exactly
    the same evidence 'rank' does. Only the mapping score -> bucket changes: a
    step function of the RANK becomes a smooth function of the VALUE.

    Standardizing ACROSS STATES (not across bars) is what makes DIR_TAU / DIR_C
    scale-free -- and is also the weakness described in the block comment above.

    require_reachable : enforce that no bucket is structurally dead (the invariant
    the reverted sign+deadzone attempt violated). For c > 0 and tau > 0 this holds
    on every state in exact arithmetic; it can only fail when the sigmoids
    SATURATE in floating point, i.e. as tau -> 0, where the rule degenerates back
    into that deadzone.
    """
    scores = composite_subset(means, feature_subset, exclude)
    n_st = len(scores)
    assert n_st >= 3, 'need at least 3 states for a 3-bucket direction map'
    _scl = DIR_SCALE if scale is None else scale
    if _scl == 'sd':
        ctr, sd = float(np.mean(scores)), float(np.std(scores))
    else:
        assert _scl == 'mad', f'unknown DIR_SCALE {_scl!r}'
        ctr = float(np.median(scores))
        sd = 1.4826 * float(np.median(np.abs(scores - ctr)))
        if sd <= 0:                       # >= half the states tied: fall back
            ctr, sd = float(np.mean(scores)), float(np.std(scores))
    z = (scores - ctr) / (sd if sd > 0 else 1.0)

    tau = DIR_TAU if tau is None else float(tau)
    c = DIR_C if c is None else float(c)
    t = max(tau, 1e-12)                      # tau -> 0 becomes a hard threshold

    w_bull = _sigmoid((z - c) / t)
    w_bear = _sigmoid((-z - c) / t)
    w_side = np.maximum(0.0, 1.0 - w_bull - w_bear)

    W = np.stack([w_bull, w_side, w_bear], axis=1)
    tot = W.sum(axis=1)
    assert (tot > 0).all(), 'a state ended up with zero weight in all three buckets'
    W = W / tot[:, None]
    # EXACT partition: after the division the row sum is 1 only to within a ulp,
    # so the residual is handed to the row's LARGEST component (>= 1/3, so
    # `1 - rest` stays safely positive and non-negativity survives).
    for i in range(n_st):
        j = int(np.argmax(W[i]))
        others = [k for k in range(3) if k != j]
        W[i, j] = 1.0 - (W[i, others[0]] + W[i, others[1]])
    assert (np.abs(W.sum(axis=1) - 1.0) <= 4 * np.finfo(float).eps).all(), \
        'per-state weights must partition to 1'
    assert (W >= 0.0).all(), 'per-state weights must be non-negative'
    if require_reachable:
        assert W[:, 1].max() > 0.0, \
            'SIDE weight is zero on every state -> SIDEWAYS unreachable (the deadzone bug)'
        assert W[:, 0].max() > 0.0, 'BULL weight is zero on every state'
        assert W[:, 2].max() > 0.0, 'BEAR weight is zero on every state'
    return W, z, scores


def hard_direction_weights(means, feature_subset, exclude=None):
    """`direction_buckets` expressed as the same one-hot (N, 3) object
    `soft_direction_weights` returns, so the two modes are one construction with
    two settings rather than two code paths. Diagnostics only -- the 'rank'
    labeling path calls `ensemble_direction_masses` itself, so the bit-for-bit
    claim is never routed through this helper."""
    d = direction_buckets(means, feature_subset, exclude)
    W = np.zeros((len(d), 3), dtype=float)
    W[d == 1, 0] = 1.0
    W[d == 0, 1] = 1.0
    W[d == -1, 2] = 1.0
    assert (W.sum(axis=1) == 1.0).all()
    return W, d


def ensemble_direction_masses_by_mode(models, Xs, feature_subset, exclude=None,
                                      direction_mode=None, tau=None, c=None, scale=None):
    """`ensemble_direction_masses` with the state -> bucket map made switchable.

    mode='rank' : DELEGATES to the unchanged function above, so it is bit-for-bit
                  today's behaviour by construction rather than by
                  re-implementation.
    mode='soft' : identical pipeline -- CAUSAL filtered posteriors, per-model
                  collapse to 3 direction columns, average across the ensemble --
                  except the collapse is `probs @ W` instead of summing the
                  columns of a hard partition.

    Averaging across the ensemble stays valid for the same reason it always did:
    direction masses are PERMUTATION-INVARIANT. Causality is untouched: W depends
    on the FITTED MEANS only (fit window) and `probs` is the forward-only
    filtered posterior.
    """
    mode = DIRECTION_MODE if direction_mode is None else direction_mode
    if mode == 'rank':
        assert tau is None and c is None and scale is None, \
            'dir_tau / dir_c / dir_scale are soft-mode knobs and do nothing under rank'
        return ensemble_direction_masses(models, Xs, feature_subset, exclude)
    assert mode == 'soft', f'unknown DIRECTION_MODE {mode!r}'
    models = list(models)
    assert len(models) >= 1, 'ensemble must contain at least one model'
    acc = np.zeros((len(Xs), 3), dtype=float)      # columns: bull, side, bear
    probs0 = None
    for i, m in enumerate(models):
        W, _z, _sc = soft_direction_weights(m.means_, feature_subset, exclude, tau, c,
                                            scale=scale)
        probs = _filtered_posteriors(m, Xs)        # CAUSAL, forward-only
        if i == 0:
            probs0 = probs
        acc += probs @ W
    acc /= len(models)
    # INVARIANT: a convex combination of simplex rows is a simplex row.
    assert np.allclose(acc.sum(axis=1), 1.0, atol=1e-9), \
        'soft ensembled direction masses must partition to 1'
    return acc[:, 0], acc[:, 1], acc[:, 2], probs0


DIR_REACH_MIN = 0.10   # a direction bucket must carry at least this much mass on
                       # SOME bar to count as reachable. Deliberately a low bar:
                       # the point is to catch a STRUCTURALLY dead bucket (the
                       # deadzone bug), not to legislate an occupancy.


def label_bars(model, Xs, dates, price, trend_raw, n_fit, feature_subset,
               exclude=None, confirm_bars=None, z_exit=None, eff_exit=None,
               dir_feats=None, bar_dir_weight=None,
               intensity_mode=None, feat_raw=None, z_enter=None,
               target_rate=None, exit_slack=None, gate_hysteresis=None,
               direction_mode=None, dir_tau=None, dir_c=None, dir_scale=None,
               escalation_during_hold=None):
    """Direction + intensity + chop-filter labeling: an inline port of the
    engine's _fit_and_classify labeling block (no repo import).

    model         : fitted GaussianHMM, OR a list/tuple of them (a seed ensemble).
                    With a list, the bull/side/bear masses are the ENSEMBLE
                    AVERAGE; a single model (or a 1-element list) reproduces the
                    old single-fit behaviour bit for bit. Nothing else about the
                    labeling changes -- the gates, the CONF_L override, the
                    output columns and their semantics are identical either way.
    Xs            : scaled features for bars 0..len(dates)-1 (scaler fit on <= n_fit)
    dates         : DatetimeIndex for those bars
    price         : full close Series (reindexed internally; efficiency is causal)
    trend_raw     : TREND_FEATURE values aligned to `dates`
    n_fit         : number of LEADING bars the model/scaler were fit on -- the trend
                    mu/sd baseline is frozen on exactly this window (no look-ahead)

    The three switchable labeling fixes (all default to the module-level config;
    the values in brackets reproduce the PRE-FIX behaviour bit for bit):

    exclude       : FIX 1, features that do not vote on direction  [()]
    confirm_bars  : FIX 2, causal confirmation delay in bars       [1]
    z_exit,
    eff_exit      : FIX 3, gate de-escalation band                 [Z_HI, EFF_HI]
    dir_feats     : FIX 4, RAW feature frame aligned to `dates` (the per-bar
                    direction score reads BAR_DIR_FEATURES out of it)
    bar_dir_weight: FIX 4, blend weight on the per-bar masses      [0.0]

    And the three switches added in this consolidation (again, the bracketed
    value reproduces the PRE-CHANGE behaviour bit for bit -- asserted against a
    frozen verbatim copy of the old function in Section 7H-viii):

    intensity_mode: FIX 5, 'frozen_z' | 'vol_norm'                 ['frozen_z']
    feat_raw      : FIX 5, raw feature frame (vol_norm reads vol_2h out of it);
                    falls back to `dir_feats`, which is the same frame at every
                    call site that supplies one
    z_enter,
    z_exit        : FIX 5/3, EXPLICIT band overrides, expressed in the units of
                    whichever intensity_mode is in force. These are no longer
                    frozen_z-only knobs -- see `gate_band`
    target_rate,
    exit_slack    : FIX 5/3, band derived by FIT-WINDOW target occupancy; valid
                    under EITHER mode
    gate_hysteresis: FIX 3 as a mode-INDEPENDENT boolean. False collapses exit
                    onto enter in either mode, which is what FIX-3-off means
                                                                   [False]
    direction_mode: FIX 6, 'rank' | 'soft'                         ['rank']
    dir_tau,
    dir_c,
    dir_scale     : FIX 6 soft-mode knobs, rejected under 'rank'
    escalation_during_hold : 'allow' | 'block' | 'demote'          ['allow']

    Returns a DataFrame indexed by `dates` with the engine's column names.
    """
    models = list(model) if isinstance(model, (list, tuple)) else [model]

    # CAUSAL decoding: filtered (forward-only) posteriors, NOT hmmlearn's
    # forward-backward predict_proba / Viterbi predict -- both of those smooth
    # bar t with bars after t, which is look-ahead in a trading label.
    #
    # Aggregate probability mass BY DIRECTION BUCKET, not by individual state: a
    # bull move split across two bullish states must not be diluted below CONF_L
    # and mislabelled SIDEWAYS. With an ensemble, those bucket masses are then
    # AVERAGED over the K fits (permutation-invariant, so this is well defined).
    # FIX 6 rides here: 'rank' delegates to the unchanged ensemble function, so
    # the default path is bit-for-bit what it was.
    _dmode = DIRECTION_MODE if direction_mode is None else direction_mode
    bull_mass, side_mass, bear_mass, probs = ensemble_direction_masses_by_mode(
        models, Xs, feature_subset, exclude, _dmode, dir_tau, dir_c, dir_scale)
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6), \
        'direction masses must partition to 1'

    # ---- FIX 4: blend in the CAUSAL PER-BAR direction ----------------------
    # The masses above are per-STATE evidence: bar t inherits the direction of
    # whichever states it sits in, and a directionally MIXED state (the classic
    # high-volatility state, which holds both sharp selloffs and sharp rallies)
    # hands the same answer to bars pointing opposite ways. The per-bar score is
    # computed from signed features only and asks the question one bar at a time.
    #
    # A convex combination of two 3-simplex points is a 3-simplex point, so the
    # partition-to-1 invariant survives untouched, and so does everything built
    # on it (prob_*, confidence, the CONF_L override, the gates, the hysteresis).
    #
    # w = 0.0 is EXACT: 1.0*m + 0.0*b == m in IEEE754 for finite non-negative m.
    _bw = BAR_DIR_WEIGHT if bar_dir_weight is None else float(bar_dir_weight)
    assert 0.0 <= _bw <= 1.0, 'BAR_DIR_WEIGHT must be in [0, 1]'
    if dir_feats is None:
        assert _bw == 0.0, \
            'bar_dir_weight > 0 requires dir_feats (the raw feature frame for these bars)'
        bar_score = np.full(len(dates), np.nan)
    else:
        assert len(dir_feats) == len(dates), 'dir_feats must be aligned 1:1 with dates'
        bar_score = bar_direction_score(dir_feats, n_fit)
        _bb, _bs, _br = bar_direction_masses(bar_score)
        bull_mass = (1.0 - _bw) * bull_mass + _bw * _bb
        side_mass = (1.0 - _bw) * side_mass + _bw * _bs
        bear_mass = (1.0 - _bw) * bear_mass + _bw * _br
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6), \
        'BLENDED direction masses must still partition to 1'

    # `states` is the FIRST ensemble member's filtered argmax. Raw state indices
    # have no ensemble-wide meaning (they permute between fits), so this column
    # is reporting-only and is never used to form a label. With K=1 it is exactly
    # the old hmm_state_int.
    states = probs.argmax(axis=1)

    # REACHABILITY -- no direction bucket may be structurally dead. This is the
    # invariant the reverted sign+deadzone attempt violated (empty side bucket ->
    # SIDEWAYS unreachable). Checked under BOTH direction modes, on the BLENDED
    # masses, i.e. on what the labels are actually formed from.
    for _nm, _m in (('BULL', bull_mass), ('SIDE', side_mass), ('BEAR', bear_mass)):
        assert float(np.max(_m)) >= DIR_REACH_MIN, \
            f'{_nm} bucket never reaches {DIR_REACH_MIN} mass on any bar -> unreachable'

    n = len(dates)
    eff = trend_efficiency(price.reindex(dates), EFF_WIN).values

    # ---- DIRECTION IS DECIDED FIRST -------------------------------------
    # The intensity gate is computed AFTER the direction now, because
    # ESCALATION_DURING_HOLD needs to know whether the emitted direction is
    # contested before it can decide whether an escalation is allowed. Nothing
    # about the pre-change computation depended on the old order: dir_raw and
    # dir_emit never read the gate, and the gate never read the direction. With
    # ESCALATION_DURING_HOLD='allow' the reorder is a pure no-op, and Section
    # 7H-viii asserts that bit-for-bit against the frozen old function.
    regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
    masses = np.stack([bull_mass, bear_mass, side_mass], axis=1)
    winner = masses.argmax(axis=1)              # 0=bull, 1=bear, 2=side (argmax, not "nonzero")

    # DIRECTION first (bull / bear / side), including the CONF_L override, ...
    dir_raw = np.select([winner == 0, winner == 1, winner == 2],
                        ['BULL', 'BEAR', 'SIDE'], default='SIDE')
    dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)

    # ... then FIX 2, the causal confirmation delay, applied to the DIRECTION.
    #
    # Why direction and not the full 5-label string: an H<->L intensity flicker
    # inside one direction would otherwise keep resetting the direction candidate
    # and can freeze the emitted label indefinitely (raw H_BULL, L_BULL, H_BULL,
    # L_BULL, ... never confirms anything at CONFIRM_BARS=2, so a clean rally
    # would stay stuck on whatever preceded it). Direction flicker is also
    # precisely the barcode the user objected to; H<->L flicker is fix 3's job.
    # confirm_bars=1 leaves dir_emit == dir_raw, i.e. the old behaviour exactly.
    dir_emit = confirm_delay(dir_raw, confirm_bars)

    # ---- THE INTENSITY GATE ---------------------------------------------
    # FIX 5: the trend-magnitude series AND the band it is graded against now
    # come from one place (`gate_band`), so INTENSITY_MODE and the enter/exit
    # band are independent knobs rather than mutually exclusive ones.
    #
    # FIX 3 lives inside that band: escalation requires |z| >= thr_enter AND
    # eff >= EFF_HI; de-escalation requires falling below the LOOSER exit band,
    # so a bar hovering at the threshold no longer flickers H/L every bar.
    # gate_hysteresis=False collapses exit onto enter in either mode, which is
    # the pre-FIX-3 memoryless test.
    _imode = INTENSITY_MODE if intensity_mode is None else intensity_mode
    _fr = feat_raw if feat_raw is not None else dir_feats
    z, thr_enter, thr_exit = gate_band(trend_raw, _fr, n_fit, _imode,
                                       z_enter, z_exit, target_rate, exit_slack,
                                       gate_hysteresis)
    assert thr_exit <= thr_enter, 'the exit band must not be tighter than the enter band'

    # ESCALATION_DURING_HOLD: a bar is CONTESTED when this bar's own evidence
    # (dir_raw) disagrees with the direction CONFIRM_BARS is holding (dir_emit).
    # Under 'allow' both masks are all-False and this is a no-op.
    _hold = ESCALATION_DURING_HOLD if escalation_during_hold is None else escalation_during_hold
    _blk, _fex, _contested = hold_masks(dir_raw, dir_emit, _hold)

    intens = intensity_state(z, eff, thr_enter, EFF_HI, thr_exit, eff_exit,
                             block_enter=_blk, force_exit=_fex)
    hi_bull = intens == 1
    hi_bear = intens == -1

    prob_cols = {
        'H_BULL':   np.where(hi_bull,  bull_mass, 0.0),
        'L_BULL':   np.where(~hi_bull, bull_mass, 0.0),
        'H_BEAR':   np.where(hi_bear,  bear_mass, 0.0),
        'L_BEAR':   np.where(~hi_bear, bear_mass, 0.0),
        'SIDEWAYS': side_mass,
    }
    # INVARIANT: the 5 prob buckets always partition the full probability mass,
    # independently of the confidence override above.
    assert np.allclose(sum(prob_cols.values()), 1.0, atol=1e-6), \
        'prob_* columns must sum to 1'

    # Intensity is then graded at bar t from the (hysteretic) gate state, so an
    # emitted H bar always clears the gates AT THAT BAR -- the chop-filter
    # invariant below is a statement about the label that is actually emitted.
    # np.full/boolean assignment rather than np.select: np.select would type the
    # result from the choicelist (<U6) and silently TRUNCATE 'SIDEWAYS' to
    # 'SIDEWA'. dtype is pinned explicitly here.
    def _compose(direction):
        st = np.full(n, 'SIDEWAYS', dtype='<U8')
        mb = direction == 'BULL'
        st[mb] = np.where(hi_bull, 'H_BULL', 'L_BULL')[mb]
        mr = direction == 'BEAR'
        st[mr] = np.where(hi_bear, 'H_BEAR', 'L_BEAR')[mr]
        return st

    regime_state = _compose(dir_emit)     # what the notebook uses everywhere
    raw_state    = _compose(dir_raw)      # pre-confirmation, diagnostics only

    # INVARIANT (chop filter), generalised for the enter/exit bands: no bar may be
    # graded H without clearing the gate that is ACTIVE for it -- the ENTER gate on
    # the first bar of an H run, the (looser) EXIT gate on a bar the run is being
    # held through. Stated against the thresholds ACTUALLY IN FORCE (thr_enter /
    # thr_exit), which is what makes it mode-independent; with hysteresis off the
    # two collapse into the single original assert.
    _ex = EFF_HI_EXIT if eff_exit is None else eff_exit
    is_h = np.isin(regime_state, ['H_BULL', 'H_BEAR'])
    if is_h.any():
        assert (eff[is_h] >= min(EFF_HI, _ex)).all(), 'H bar below the EFF exit band -> chop filter bypassed'
        assert (np.abs(z[is_h]) >= thr_exit).all(), 'H bar below the intensity exit band -> magnitude gate bypassed'
        # and every ESCALATION -- the bar on which the gate state machine turned ON,
        # which is where the ENTER band must have been cleared. (The bar an emitted
        # H *label* run starts on is NOT the right anchor: the direction can flip to
        # BULL several bars into an already-escalated stretch, and that bar only
        # owes the exit band.)
        _prev_i = np.concatenate(([0], intens[:-1]))
        _on = np.flatnonzero((intens != 0) & (intens != _prev_i))   # incl. +1 -> -1 flips
        assert (eff[_on] >= EFF_HI).all(), 'H escalation below EFF_HI -> enter gate bypassed'
        assert (np.abs(z[_on]) >= thr_enter).all(), \
            'H escalation below the intensity enter threshold -> enter gate bypassed'
        # ESCALATION_DURING_HOLD: under 'block'/'demote' no escalation may BEGIN on
        # a contested bar, and under 'demote' no H may be emitted on one at all.
        if _hold in ('block', 'demote'):
            assert not _contested[_on].any(), \
                'a NEW escalation fired on a contested bar -> ESCALATION_DURING_HOLD bypassed'
        if _hold == 'demote':
            assert not (is_h & _contested).any(), \
                'an H label survived on a contested bar under ESCALATION_DURING_HOLD=demote'
    assert set(np.unique(regime_state)).issubset(set(REGIME_LABELS)), 'unknown label emitted'

    out = pd.DataFrame({
        'tactical_regime_state':      regime_state,
        'tactical_regime_confidence': regime_confidence,
        'hmm_state_int':              states.astype(int),
        'prob_H_BULL':                prob_cols['H_BULL'],
        'prob_L_BULL':                prob_cols['L_BULL'],
        'prob_SIDEWAYS':              prob_cols['SIDEWAYS'],
        'prob_L_BEAR':                prob_cols['L_BEAR'],
        'prob_H_BEAR':                prob_cols['H_BEAR'],
        'trend_z':                    z,
        'trend_efficiency':           eff,
        # diagnostics for Section 7H (never inputs to anything):
        'gate_intensity':             intens,
        'regime_state_raw':           raw_state,
        'bar_dir_score':              bar_score,
        # ESCALATION_DURING_HOLD diagnostics (never inputs to anything).
        # `contested` keeps the prototype's column name so the two artifacts
        # can be diffed directly.
        'dir_raw':                    dir_raw,
        'dir_emit':                   dir_emit,
        'contested':                  _contested,
    }, index=dates)
    out.index.name = 'date'
    out.attrs['intensity_mode'] = _imode
    out.attrs['direction_mode'] = _dmode
    out.attrs['escalation_during_hold'] = _hold
    out.attrs['thr_enter'] = float(thr_enter)
    out.attrs['thr_exit'] = float(thr_exit)
    return out


# ---------------------------------------------------------------------------
# THE FROZEN PRE-CHANGE REFERENCE.
#
# `label_bars_legacy` is a VERBATIM copy of `label_bars` as it stood BEFORE this
# consolidation -- before INTENSITY_MODE, DIRECTION_MODE, ESCALATION_DURING_HOLD
# and the direction-before-intensity reorder. It exists for exactly one purpose:
# Section 7H-viii runs both functions over a matrix of argument combinations and
# asserts BIT-FOR-BIT equality of every emitted column whenever the new switches
# sit at their OFF values. That turns "these switches are no-ops when off" from a
# claim in a comment into a test.
#
# It is never called by the pipeline. Do not "improve" it -- its whole value is
# that it is frozen.
# ---------------------------------------------------------------------------
def label_bars_legacy(model, Xs, dates, price, trend_raw, n_fit, feature_subset,
                      exclude=None, confirm_bars=None, z_exit=None, eff_exit=None,
                      dir_feats=None, bar_dir_weight=None):
    models = list(model) if isinstance(model, (list, tuple)) else [model]

    bull_mass, side_mass, bear_mass, probs = \
        ensemble_direction_masses(models, Xs, feature_subset, exclude)
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6)

    _bw = BAR_DIR_WEIGHT if bar_dir_weight is None else float(bar_dir_weight)
    assert 0.0 <= _bw <= 1.0
    if dir_feats is None:
        assert _bw == 0.0
        bar_score = np.full(len(dates), np.nan)
    else:
        assert len(dir_feats) == len(dates)
        bar_score = bar_direction_score(dir_feats, n_fit)
        _bb, _bs, _br = bar_direction_masses(bar_score)
        bull_mass = (1.0 - _bw) * bull_mass + _bw * _bb
        side_mass = (1.0 - _bw) * side_mass + _bw * _bs
        bear_mass = (1.0 - _bw) * bear_mass + _bw * _br
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6)

    states = probs.argmax(axis=1)

    trend_raw = np.asarray(trend_raw, dtype=float)
    trend_mu = float(np.mean(trend_raw[:n_fit]))
    trend_sd = float(np.std(trend_raw[:n_fit]))
    z = (trend_raw - trend_mu) / (trend_sd if trend_sd > 0 else 1.0)

    eff = trend_efficiency(price.reindex(dates), EFF_WIN).values
    intens = intensity_state(z, eff, Z_HI, EFF_HI, z_exit, eff_exit)
    hi_bull = intens == 1
    hi_bear = intens == -1

    n = len(dates)
    prob_cols = {
        'H_BULL':   np.where(hi_bull,  bull_mass, 0.0),
        'L_BULL':   np.where(~hi_bull, bull_mass, 0.0),
        'H_BEAR':   np.where(hi_bear,  bear_mass, 0.0),
        'L_BEAR':   np.where(~hi_bear, bear_mass, 0.0),
        'SIDEWAYS': side_mass,
    }
    assert np.allclose(sum(prob_cols.values()), 1.0, atol=1e-6)

    regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
    masses = np.stack([bull_mass, bear_mass, side_mass], axis=1)
    winner = masses.argmax(axis=1)

    dir_raw = np.select([winner == 0, winner == 1, winner == 2],
                        ['BULL', 'BEAR', 'SIDE'], default='SIDE')
    dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)
    dir_emit = confirm_delay(dir_raw, confirm_bars)

    def _compose(direction):
        st = np.full(n, 'SIDEWAYS', dtype='<U8')
        mb = direction == 'BULL'
        st[mb] = np.where(hi_bull, 'H_BULL', 'L_BULL')[mb]
        mr = direction == 'BEAR'
        st[mr] = np.where(hi_bear, 'H_BEAR', 'L_BEAR')[mr]
        return st

    regime_state = _compose(dir_emit)
    raw_state    = _compose(dir_raw)

    return pd.DataFrame({
        'tactical_regime_state':      regime_state,
        'tactical_regime_confidence': regime_confidence,
        'hmm_state_int':              states.astype(int),
        'prob_H_BULL':                prob_cols['H_BULL'],
        'prob_L_BULL':                prob_cols['L_BULL'],
        'prob_SIDEWAYS':              prob_cols['SIDEWAYS'],
        'prob_L_BEAR':                prob_cols['L_BEAR'],
        'prob_H_BEAR':                prob_cols['H_BEAR'],
        'trend_z':                    z,
        'trend_efficiency':           eff,
        'gate_intensity':             intens,
        'regime_state_raw':           raw_state,
        'bar_dir_score':              bar_score,
    }, index=dates)


print('features + direction/intensity labeling core ready')
print(f'  gate      : INTENSITY_MODE={INTENSITY_MODE!r}  band via gate_band() '
      f'(mode and band are independent knobs)')
print(f'  direction : DIRECTION_MODE={DIRECTION_MODE!r}   hold policy: '
      f'ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}')
print('  label_bars_legacy (frozen pre-change copy) available for the 7H-viii equivalence test')

In [ ]:
# ==========================================================================
# PLOT HELPERS -- INLINED VERBATIM (master code cell 9).
# regime_blocks / shade_bands / shade_regimes / set_price_ylim / regime_legend.
# These are the MASTER NOTEBOOK regime-background style used by figure 4.
# ==========================================================================
def regime_blocks(series):
    """[(label, start_ts, end_ts), ...] contiguous runs of the same label."""
    vals, idx = series.values, series.index
    blocks, start = [], 0
    for i in range(1, len(vals)):
        if vals[i] != vals[i - 1]:
            blocks.append((vals[start], idx[start], idx[i - 1]))
            start = i
    blocks.append((vals[start], idx[start], idx[-1]))
    return blocks


def shade_bands(ax, spans, alpha=0.35, zorder=1):
    """Paint (label, x0, x1) spans as ONE PolyCollection PER LABEL.

    Replaces a per-block `ax.axvspan` loop. On this data a chart has 500+
    contiguous regime blocks, so the loop built 500+ individual Patch artists per
    axes and ~13 figures paid for it; this builds at most len(REGIME_LABELS)
    collections instead, with the same geometry.

    Visually identical, by construction rather than by eye:
      * the x-ranges come from the SAME `regime_blocks` output, unchanged;
      * `broken_barh` with `ax.get_xaxis_transform()` is the same blended
        transform `axvspan` uses -- x in DATA coordinates, y in AXES fraction
        0..1 -- so bands span the full height and ignore the y data limits
        exactly as axvspan did;
      * colour, alpha, linewidth=0 and zorder=1 are the axvspan values.

    Grouping by label is safe because `regime_blocks` returns DISJOINT spans, so
    no two bands overlap and the draw order between them cannot matter.

    ------------------------------------------------------------------------
    THE Y-AXIS BUG THIS FIXES (user-visible; NOT reproducible on every
    matplotlib, so it is fixed structurally rather than by chasing a repro).
    ------------------------------------------------------------------------
    On the user's Kaggle matplotlib the shaded price panels came out with the
    y-axis dragged down to 0, squashing the price line into the top fifth of the
    panel. The cause is the blended transform: the band geometry is y = 0..1 in
    AXES-FRACTION coordinates, but `broken_barh` -> `add_collection` defaults to
    `autolim=True`, and an older matplotlib folds the collection's raw y-extent
    (those literal 0 and 1) into the axes DATA limits before the transform is
    considered. The autoscaler then has to fit both `[0, 1]` and `[24000, 26000]`
    and produces `[0, 26000]`.

    Fixed two ways, deliberately belt-and-braces:
      1. HERE -- build the PolyCollection directly and add it with
         `autolim=False`, so it cannot contribute to the datalim on ANY
         matplotlib version. This is the structural fix.
      2. At every call site -- `set_price_ylim` sets the y-limits EXPLICITLY from
         the plotted series, so the autoscaler is never consulted at all.
    Each of the two alone is sufficient; together the panel cannot regress.

    The speed optimization is NOT reverted: this still builds at most
    len(REGIME_LABELS) collections per axes, not one Patch per block.
    """
    import matplotlib.dates as _mdates
    from matplotlib.collections import PolyCollection
    by_lab = {}
    for lb, d0, d1 in spans:
        x0, x1 = _mdates.date2num(d0), _mdates.date2num(d1)
        by_lab.setdefault(lb, []).append((x0, x1))
    for lb, xr in by_lab.items():
        verts = [[(x0, 0.0), (x1, 0.0), (x1, 1.0), (x0, 1.0)] for x0, x1 in xr]
        coll = PolyCollection(verts,
                              facecolors=REGIME_COLORS.get(lb, '#808080'),
                              alpha=alpha, linewidths=0, zorder=zorder)
        # x in DATA coords, y in AXES fraction 0..1 -- the same blended transform
        # axvspan and broken_barh use, so the bands still span the full height.
        coll.set_transform(ax.get_xaxis_transform())
        ax.add_collection(coll, autolim=False)     # <-- cannot touch the datalim


def shade_regimes(ax, series, alpha=0.35):
    shade_bands(ax, regime_blocks(series), alpha=alpha)


# Registry of every shaded panel whose y-limits were set explicitly, so Section
# 7Z can assert -- once, centrally -- that each one brackets its own series and
# excludes 0. A panel that forgot to call this simply never gets checked, so the
# registry is printed with its expected count too.
YLIM_CHECKS = []


def set_price_ylim(ax, series, pad=0.03, tag=''):
    """Set y-limits EXPLICITLY from the plotted series and record the check.

    Never leaves a shaded price/VIX panel to the autoscaler. `pad` is a fraction
    of the series range (falling back to a fraction of the level, then to 1.0,
    for a degenerate flat series).
    """
    v = np.asarray(series, dtype=float)
    v = v[np.isfinite(v)]
    assert v.size, f'set_price_ylim got no finite values ({tag})'
    lo, hi = float(v.min()), float(v.max())
    m = (hi - lo) * pad or abs(hi) * pad or 1.0
    ax.set_ylim(lo - m, hi + m)
    YLIM_CHECKS.append((tag, ax, lo, hi))
    return lo, hi


SPLIT_STYLE = dict(color='blue', linestyle='--', linewidth=1.2, zorder=5)
SPLIT_LABEL = 'train/test split'


def mark_split(ax, index, split_ts):
    """Draw the anchored train/test boundary, if it falls inside this panel.

    Returns the legend handle when the line was drawn and None when the panel's
    window does not contain the split (the zoom panels), so a caller can add the
    legend entry only where there is actually a line to explain.

    THE SPLIT IS A BACKTEST DEVICE. It marks where the fit window ended so that
    bars to its right can be scored on data the model never saw. A LIVE engine
    has no such boundary: it fits on all history to date and classifies the next
    bar. Nothing to the left of this line is "less real" -- it is simply
    in-sample, and therefore not evidence.
    """
    if split_ts is None or len(index) == 0:
        return None
    if not (index[0] <= split_ts <= index[-1]):
        return None
    ax.axvline(split_ts, **SPLIT_STYLE)
    return plt.Line2D([0], [0], color=SPLIT_STYLE['color'],
                      ls=SPLIT_STYLE['linestyle'], label=SPLIT_LABEL)


def regime_legend(ax, loc='upper left', extra=None, **kw):
    handles = [mpatches.Patch(color=REGIME_COLORS[r], alpha=0.7, label=r) for r in REGIME_LABELS]
    if extra:
        handles += extra
    ax.legend(handles=handles, loc=loc, fontsize=8, ncol=3, **kw)


def synth_tag():
    return "  [SYNTHETIC DATA - illustrative only]" if TAC_SYNTH else "  [real yfinance data]"

In [ ]:
# ---------------------------------------------------------------------------
# PROVENANCE BANNER + the two standing decrees, asserted rather than promised.
# ---------------------------------------------------------------------------
import itertools, time, contextlib
import matplotlib.dates as mdates
from scipy.stats import spearmanr

T_START = time.time()

if TAC_SYNTH:
    print('#' * 100)
    for _ in range(3):
        print('#   SYNTHETIC - ILLUSTRATIVE ONLY   ' * 2)
    print('#' * 100)
    print('#  The yfinance fetch FAILED (it is firewalled in some sandboxes).')
    print('#  Every chart and number below is generated from a synthetic GBM series.')
    print('#  It is a PLUMBING TEST ONLY. It says NOTHING about real Nifty regimes,')
    print('#  about real fit stability, and NOTHING about real transition lag.')
    print('#  Re-run on Kaggle, where yfinance works, before believing any verdict.')
    print('#' * 100)
else:
    print('=' * 100)
    print('REAL yfinance 2h data. Verdicts below are decision-grade.')
    print('=' * 100)

# DECREE 1 -- BAR_DIR_WEIGHT is settled at 0.0 and is not a free variable here.
assert BAR_DIR_WEIGHT == 0.0, 'BAR_DIR_WEIGHT must be 0.0 for this notebook'
W_FIXED = 0.0
print(f'\nDECREE: BAR_DIR_WEIGHT is FIXED at {W_FIXED} (direction is 100% HMM). '
      f'Not swept. Asserted at every labelling call.')

print(f'bars={len(nifty)}   span {nifty.index[0]:%Y-%m-%d} -> {nifty.index[-1]:%Y-%m-%d}')
print(f'shipped labeling knobs: CONFIRM_BARS={CONFIRM_BARS}  ENSEMBLE_K={ENSEMBLE_K}  '
      f'ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}  '
      f'INTENSITY_MODE={INTENSITY_MODE!r}  DIRECTION_MODE={DIRECTION_MODE!r}')
print(f'shipped config       : {ADOPTED_CONFIG_NAME}')

## 2. The metrics — S, L, W — and the ZigZag leakage tripwire

### The look-ahead question, answered before the code

The ZigZag **uses future prices on purpose**. It is a *retrospective evaluation*
device, exactly like a forward return: it grades a decision that was already made.
The protocol permits that and requires one thing in exchange — **it must never
touch a label**.

That is enforced three ways, all of them runtime asserts, not comments:

1. **Phase ordering.** All labelling happens in the LABEL PHASE; the ZigZag is not
   computed until the EVAL PHASE. `label_bars` is wrapped by a guard that asserts
   `ZZ_CALLS == 0` — i.e. *no label was ever produced after a ZigZag existed*.
2. **Object identity.** The guard scans every positional and keyword argument of
   every `label_bars` call and refuses any object that is a ZigZag output.
3. **Memory aliasing.** For array arguments it additionally asserts
   `np.shares_memory(arg, zz) is False` against every ZigZag array, catching a view
   or a slice that identity alone would miss.

The same wrapper asserts `bar_dir_weight == 0.0` on every single call, so
`BAR_DIR_WEIGHT` cannot drift inside a sweep.

In [ ]:
# ===========================================================================
# METRIC PARAMETERS -- ALL DECLARED HERE, BEFORE ANY NUMBER EXISTS.
# ===========================================================================
R_SEED_SETS   = 4       # protocol: R = 4 DISJOINT seed sets. NOT reducible.
ZZ_PCT        = 2.0     # protocol: ZigZag reversal threshold, 2.0%
W_GUARD_MAX   = 25.0    # G3
OCC_MIN_PCT   = 3.0     # G1 / G2
OCC_MAX_PCT   = 50.0    # G1
SIDEWAYS_MAX  = 50.0    # G4

# S2 -- log-likelihood pruning threshold, expressed as a FRACTION OF THE LL SPREAD
# of the ensemble. Declared here, not tuned to a result. A member is kept when
#     (best_ll - ll) <= S2_LL_FRAC * (best_ll - worst_ll)
S2_LL_FRAC    = 0.25

# L2 -- adaptive confirmation. The "strong signal" threshold is declared as a RATE
# over the FIT WINDOW (the same device H_TARGET_RATE already uses), not as a raw
# number tuned after seeing lag. Strictly causal: it is a fit-window constant.
L2_FAST_RATE  = 0.25    # the top 25% of fit-window bars by |bull_mass - bear_mass|
                        # get 1-bar confirmation; everything else keeps 2.

print(f'R = {R_SEED_SETS} disjoint seed sets   ZigZag = {ZZ_PCT}%   '
      f'G3 W<={W_GUARD_MAX}   G1 occ in [{OCC_MIN_PCT},{OCC_MAX_PCT}]%   '
      f'G4 SIDEWAYS<={SIDEWAYS_MAX}%')
print(f'S2_LL_FRAC={S2_LL_FRAC}   L2_FAST_RATE={L2_FAST_RATE}   '
      '(both declared before any result)')

In [ ]:
# ===========================================================================
# THE ZIGZAG -- LABEL-BLIND, RETROSPECTIVE. Sees prices only; never a label.
# ===========================================================================
ZZ_CALLS = 0            # how many times a ZigZag has been computed
ZZ_IDS = set()          # id() of every object a ZigZag has ever returned
ZZ_ARRAYS = []          # the arrays themselves, for np.shares_memory checks


def zigzag_pivots(px, pct=ZZ_PCT):
    '''Indices of ZigZag pivots on a close series, `pct`% reversal threshold.

    RETROSPECTIVE BY CONSTRUCTION: a pivot at bar i is only confirmed once price
    has moved pct% away from it, which happens at some bar > i. That is exactly
    the look-ahead this function is allowed to have and `label_bars` is not.

    Takes a bare float array in and returns a bare int array out -- it has no
    access to a label, a mass, a model or a feature frame.
    '''
    global ZZ_CALLS
    px = np.asarray(px, dtype=float)
    n = len(px)
    if n < 3:
        out = np.array([], dtype=int)
    else:
        hi_i = lo_i = 0
        d, start, piv, ext_i = 0, None, [], 0
        for i in range(1, n):
            if px[i] > px[hi_i]:
                hi_i = i
            if px[i] < px[lo_i]:
                lo_i = i
            if (px[hi_i] - px[i]) / px[hi_i] * 100.0 >= pct:
                d, piv, ext_i, start = -1, [hi_i], i, i
                break
            if (px[i] - px[lo_i]) / px[lo_i] * 100.0 >= pct:
                d, piv, ext_i, start = +1, [lo_i], i, i
                break
        if d == 0:
            out = np.array([], dtype=int)
        else:
            for i in range(start + 1, n):
                if d > 0:
                    if px[i] >= px[ext_i]:
                        ext_i = i
                    elif (px[ext_i] - px[i]) / px[ext_i] * 100.0 >= pct:
                        piv.append(ext_i); d, ext_i = -1, i
                else:
                    if px[i] <= px[ext_i]:
                        ext_i = i
                    elif (px[i] - px[ext_i]) / px[ext_i] * 100.0 >= pct:
                        piv.append(ext_i); d, ext_i = +1, i
            if ext_i != piv[-1]:
                piv.append(ext_i)          # final, PROVISIONAL pivot
            out = np.asarray(piv, dtype=int)

    ZZ_CALLS += 1
    ZZ_IDS.add(id(out))
    ZZ_ARRAYS.append(out)
    return out


def zigzag_swings(px, pct=ZZ_PCT):
    '''[(i0, i1, +1/-1), ...] -- consecutive pivot pairs that clear `pct`.

    The last (provisional) leg is kept only if it actually cleared the threshold,
    so a half-formed leg at the right edge cannot inflate or deflate lag.
    '''
    px = np.asarray(px, dtype=float)
    piv = zigzag_pivots(px, pct)
    sw = []
    for a, b in zip(piv[:-1], piv[1:]):
        move = (px[b] - px[a]) / px[a] * 100.0
        if abs(move) >= pct:
            sw.append((int(a), int(b), 1 if move > 0 else -1))
    out = sw
    ZZ_IDS.add(id(out))
    return out


# ---------------------------------------------------------------------------
# THE LEAKAGE TRIPWIRE. `label_bars` is SHADOWED by a guard; every later call in
# this notebook goes through it. Three independent checks, plus the w=0 decree.
# ---------------------------------------------------------------------------
_label_bars_raw = label_bars
LABEL_CALLS = 0
LABEL_PHASE_OPEN = True     # flipped shut once the EVAL phase starts


@contextlib.contextmanager
def reopen_label_phase(why):
    '''EXPLICITLY and LOUDLY reopen the label phase after the ZigZag exists.

    Used once, for the combination arms, whose identity cannot be known until the
    single interventions have been measured. Only the PHASE-ORDERING check is
    relaxed; checks 2, 3 and 4 (object identity, memory aliasing, w == 0.0) stay
    armed throughout, and the reopening prints itself so it can never be a silent
    hole in the proof.
    '''
    global LABEL_PHASE_OPEN
    print('!' * 78)
    print(f'!! LABEL PHASE EXPLICITLY REOPENED: {why}')
    print('!! ordering check relaxed; identity / aliasing / w==0 checks STAY ARMED')
    print('!' * 78)
    LABEL_PHASE_OPEN = True
    try:
        yield
    finally:
        LABEL_PHASE_OPEN = False
        print('!! LABEL PHASE CLOSED AGAIN -- ordering check re-armed.')


def label_bars(*args, **kwargs):
    '''Guarded shadow of the engine's label_bars. Adds asserts, changes nothing.'''
    global LABEL_CALLS
    assert ZZ_CALLS == 0 or LABEL_PHASE_OPEN, (
        'LEAKAGE: a label was produced AFTER a ZigZag had been computed. All '
        'labelling must complete in the LABEL PHASE, before any ZigZag exists.')
    _bw = kwargs.get('bar_dir_weight', BAR_DIR_WEIGHT)
    assert _bw == 0.0, f'BAR_DIR_WEIGHT drifted to {_bw} inside a labelling call'
    for v in list(args) + list(kwargs.values()):
        assert id(v) not in ZZ_IDS, 'LEAKAGE: a ZigZag output was passed to label_bars'
        if isinstance(v, np.ndarray):
            for z in ZZ_ARRAYS:
                assert not np.shares_memory(v, z), \
                    'LEAKAGE: a label_bars argument aliases ZigZag memory'
    LABEL_CALLS += 1
    return _label_bars_raw(*args, **kwargs)


print('ZigZag + leakage tripwire ready.')
print('  check 1  phase ordering  : label_bars asserts ZZ_CALLS == 0')
print('  check 2  object identity : every argument checked against ZZ_IDS')
print('  check 3  memory aliasing : np.shares_memory vs every ZigZag array')
print('  check 4  decree          : bar_dir_weight == 0.0 on every call')

In [ ]:
# ===========================================================================
# S / L / W  -- the three metrics, one function each. No composite anywhere.
# ===========================================================================
DIR_OF_LABEL = {'H_BULL': 1, 'L_BULL': 1, 'SIDEWAYS': 0, 'L_BEAR': -1, 'H_BEAR': -1}


def emitted_direction(labels):
    '''5-label string array -> +1 bull / 0 sideways / -1 bear.'''
    return np.array([DIR_OF_LABEL[x] for x in np.asarray(labels)], dtype=int)


def metric_S(label_sets):
    '''S = mean pairwise 5-LABEL agreement (%) across R seed sets.

    Returns (S_mean, pairwise_matrix RxR, list_of_pairwise_values).
    The FULL matrix is returned because the protocol requires it reported, not
    just the mean.
    '''
    R = len(label_sets)
    M = np.full((R, R), np.nan)
    vals = []
    for i in range(R):
        M[i, i] = 100.0
        for j in range(i + 1, R):
            a = np.asarray(label_sets[i]); b = np.asarray(label_sets[j])
            assert len(a) == len(b), 'label sets must be the same length'
            v = 100.0 * float(np.mean(a == b))
            M[i, j] = M[j, i] = v
            vals.append(v)
    return float(np.mean(vals)), M, vals


def metric_L(labels, swings):
    '''L per ZigZag swing = bars from swing START to the first correctly
    directed EMITTED label inside the swing.

    An UNMATCHED swing (never labelled correctly anywhere inside it) scores the
    FULL swing length -- the worst case, per protocol. It is not dropped.

    Returns (median, p75, per_swing_array, n_unmatched).
    '''
    d = emitted_direction(labels)
    lags, unmatched = [], 0
    for i0, i1, sgn in swings:
        seg = d[i0:i1 + 1]
        hit = np.flatnonzero(seg == sgn)
        if hit.size:
            lags.append(int(hit[0]))
        else:
            lags.append(int(i1 - i0))
            unmatched += 1
    if not lags:
        return np.nan, np.nan, np.array([]), 0
    a = np.asarray(lags, dtype=float)
    return float(np.median(a)), float(np.percentile(a, 75)), a, unmatched


def metric_W(labels):
    '''W = 5-label switches per 100 bars.'''
    a = np.asarray(labels)
    if len(a) < 2:
        return 0.0
    return 100.0 * float(np.sum(a[1:] != a[:-1])) / (len(a) - 1)


def occupancy_pct(labels):
    a = np.asarray(labels)
    return {lb: 100.0 * float(np.mean(a == lb)) for lb in REGIME_LABELS}


# ---------------------------------------------------------------------------
# GUARDS G1..G4. One implementation, used by real arms AND by the degeneracy
# stub in section 3, so the test exercises the shipping code path.
# ---------------------------------------------------------------------------
def evaluate_guards(occ, W):
    '''{name: (bool_pass, reason)} for the four protocol guards.'''
    g = {}
    bad_hi = [l for l in REGIME_LABELS if occ.get(l, 0.0) > OCC_MAX_PCT]
    bad_lo = [l for l in REGIME_LABELS if occ.get(l, 0.0) < OCC_MIN_PCT]
    g['G1'] = (not bad_hi and not bad_lo,
               'ok' if not (bad_hi or bad_lo)
               else f'outside [{OCC_MIN_PCT},{OCC_MAX_PCT}]%: '
                    + ','.join(sorted(set(bad_hi + bad_lo))))
    g['G2'] = (not bad_lo, 'ok' if not bad_lo else 'collapsed: ' + ','.join(bad_lo))
    g['G3'] = (W <= W_GUARD_MAX, f'W={W:.2f} vs max {W_GUARD_MAX}')
    g['G4'] = (occ.get('SIDEWAYS', 0.0) <= SIDEWAYS_MAX,
               f"SIDEWAYS={occ.get('SIDEWAYS', 0.0):.1f}% vs max {SIDEWAYS_MAX}%")
    return g


def guards_pass(g):
    return all(v[0] for v in g.values())


print('metrics ready: metric_S (pairwise matrix), metric_L (median/p75, unmatched '
      '= full swing length), metric_W, evaluate_guards (G1-G4).')
print('NO composite score is defined anywhere in this notebook -- by design.')

## 3. Machinery self-tests — including **driving G4 with a degenerate stub**

Four tests, all of which must pass before a single arm is fitted.

1. **ZigZag correctness** on a hand-built sawtooth with known pivots.
2. **`confirm_delay_dyn` equivalence** — the generalised (per-bar) confirmation
   scanner must reproduce the engine's frozen `confirm_delay` bit-for-bit when its
   requirement is held constant. This is the "every new switch has an OFF value
   asserted against the frozen original" rule.
3. **G4 bites.** A stub that forces an all-`SIDEWAYS` labelling is pushed through
   the *same* summarising and guard code the real arms use. It scores a perfect
   `S = 100.00%` — which is precisely how S is gamed — and must be **VOIDED**.
4. **G3 bites** on a synthetic high-whipsaw labelling.

In [ ]:
# ---------------------------------------------------------------------------
# The generalised confirmation scanner. Structurally identical to the engine's
# confirm_delay; the ONLY change is that the required run length is asked for
# per bar via need_fn(t, prev_emitted, candidate) instead of being a constant.
# STILL STRICTLY CAUSAL: out[t] depends on raw[0..t] and out[t-1] only.
# ---------------------------------------------------------------------------
def confirm_delay_dyn(raw, need_fn):
    raw = np.asarray(raw)
    n = len(raw)
    out = np.empty(n, dtype=raw.dtype)
    if n == 0:
        return out
    out[0] = raw[0]
    cand, run = raw[0], 0
    for t in range(1, n):
        if raw[t] == out[t - 1]:
            out[t] = raw[t]
            cand, run = raw[t], 0
        else:
            if raw[t] == cand:
                run += 1
            else:
                cand, run = raw[t], 1
            if run >= int(need_fn(t, out[t - 1], cand)):
                out[t] = cand
                run = 0
            else:
                out[t] = out[t - 1]
    return out


@contextlib.contextmanager
def confirm_policy(need_fn):
    '''Temporarily route label_bars' confirmation through `need_fn`.

    label_bars resolves `confirm_delay` from the notebook globals at call time,
    so rebinding that name here changes the confirmation rule WITHOUT editing,
    copying or forking a single line of the inlined engine. Restored on exit.
    '''
    g = globals()
    old = g['confirm_delay']

    def _patched(raw, confirm_bars=None):
        return confirm_delay_dyn(raw, need_fn)

    g['confirm_delay'] = _patched
    try:
        yield
    finally:
        g['confirm_delay'] = old


# ---- TEST 1: ZigZag on a known sawtooth -----------------------------------
_saw = np.array([100, 101, 100.5, 105, 104, 101, 100, 103, 108, 107, 102, 106.])
_sw = zigzag_swings(_saw, pct=2.0)
assert len(_sw) >= 3, f'zigzag found too few swings on the sawtooth: {_sw}'
for _a, _b, _s in _sw:
    _mv = (_saw[_b] - _saw[_a]) / _saw[_a] * 100
    assert _a < _b and np.sign(_mv) == _s and abs(_mv) >= 2.0, f'bad swing {(_a,_b,_s)}'
assert [s for _, _, s in _sw] == [(-1) ** k * _sw[0][2] for k in range(len(_sw))], \
    'zigzag swing directions must alternate'
print(f'TEST 1 PASS  zigzag: {len(_sw)} alternating swings, each >= 2.0%: {_sw}')

# The unit test above ran the ZigZag on a 12-point TOY array that is not the
# price series and never enters the pipeline. Its bookkeeping is cleared here,
# explicitly and in the open, so the leakage tripwire starts the label phase from
# a true zero rather than from a self-test artefact.
ZZ_CALLS = 0
ZZ_IDS.clear()
ZZ_ARRAYS.clear()
print('             (ZigZag bookkeeping reset after the toy-array unit test; the '
      'tripwire now starts from ZZ_CALLS = 0)')

# ---- TEST 2: confirm_delay_dyn == confirm_delay when the need is constant ---
_rng = np.random.default_rng(7)
_raw = _rng.choice(np.array(['BULL', 'BEAR', 'SIDE']), size=4000)
for _cb in (1, 2, 3, 5):
    _ref = confirm_delay(_raw, _cb)
    _got = confirm_delay_dyn(_raw, lambda t, p, c, _n=_cb: _n)
    assert np.array_equal(_ref, _got), f'confirm_delay_dyn diverges at confirm_bars={_cb}'
print('TEST 2 PASS  confirm_delay_dyn reproduces the frozen confirm_delay '
      'bit-for-bit at confirm_bars = 1, 2, 3, 5 (its OFF value)')

# ---- TESTS 3 & 4 are deferred to section 6, where summarise_arm exists ------
print('TESTS 3 & 4 (G4 / G3 bite) run in section 6, against the REAL guard path.')

## 4. Fit harness — R = 4 disjoint seed sets

`R = 4` is the whole reason S means anything, so it is **never** the thing that
gets cut for runtime. Seed set *r* uses seeds `BASE_SEED + r*K + 0..K-1`, which are
disjoint by construction and asserted to be.

Every arm is (config × ensemble policy × labelling policy). Fits are **cached** by
`(features, N, cov, K, R, init)` so that the eight labelling-side interventions and
every combination reuse the baseline's fits instead of refitting — the refits that
remain are exactly the ones that *change the fit*.

In [ ]:
# ===========================================================================
# FEATURES + SCALING (per feature subset), and the seed-set fitter.
# ===========================================================================
FEAT_DF = build_features(nifty, vix, LOOKBACK_SCALE)
DATES = FEAT_DF.index
CLOSE = nifty.reindex(DATES).values
TREND_RAW = FEAT_DF[TREND_FEATURE].values
N_BARS = len(DATES)
N_FIT = max(int(N_BARS * TRAIN_FRACTION), 50)
print(f'{N_BARS} feature bars; anchored fit window = {N_FIT} bars '
      f'({TRAIN_FRACTION:.0%})  {DATES[0]:%Y-%m-%d} -> {DATES[-1]:%Y-%m-%d}')

_XS_CACHE = {}


def scaled_X(features):
    key = tuple(features)
    if key not in _XS_CACHE:
        raw = FEAT_DF[list(features)].values
        sc = StandardScaler().fit(raw[:N_FIT])
        _XS_CACHE[key] = sc.transform(raw)
    return _XS_CACHE[key]


def seed_sets(K, R=R_SEED_SETS, base=BASE_SEED):
    ss = [list(range(base + r * K, base + r * K + K)) for r in range(R)]
    flat = [s for st in ss for s in st]
    assert len(set(flat)) == len(flat), 'seed sets must be DISJOINT'
    return ss


def fit_deterministic(X, N, cov, n_iter=None):
    '''S4 -- SEED-FREE initialisation.

    means from a fixed-seed k-means, sorted deterministically; startprob, transmat
    and covars set to fixed values. `init_params=''` means hmmlearn initialises
    NOTHING, so `random_state` cannot enter the fit at all. Consequence, stated
    plainly: every seed produces the IDENTICAL model, so S is 100% BY
    CONSTRUCTION. That is a real property, not an achievement -- see section 7.
    '''
    from sklearn.cluster import KMeans
    n_iter = HMM_ITER if n_iter is None else n_iter
    km = KMeans(n_clusters=N, n_init=10, random_state=0).fit(X)
    means = km.cluster_centers_[np.argsort(km.cluster_centers_[:, 0])]
    m = hmm.GaussianHMM(n_components=N, covariance_type=cov, n_iter=n_iter,
                        init_params='', params='stmc')
    m.startprob_ = np.full(N, 1.0 / N)
    tm = np.full((N, N), 0.1 / max(N - 1, 1))
    np.fill_diagonal(tm, 0.9)
    m.transmat_ = tm
    m.means_ = means
    d = X.shape[1]
    if cov == 'full':
        m.covars_ = np.tile(np.cov(X.T) + 1e-3 * np.eye(d), (N, 1, 1))
    else:
        m.covars_ = np.tile(X.var(axis=0) + 1e-6, (N, 1))
    m.fit(X)
    return m


FIT_CACHE = {}
FIT_COUNT = 0


def get_fits(features, N, cov, K, init='seeded'):
    '''[models_for_seed_set_0, ..., models_for_seed_set_{R-1}]. Cached.'''
    global FIT_COUNT
    key = (tuple(features), N, cov, K, R_SEED_SETS, init)
    if key in FIT_CACHE:
        return FIT_CACHE[key]
    X = scaled_X(features)[:N_FIT]
    sets = seed_sets(K)
    out = []
    for sd in sets:
        if init == 'det':
            m = fit_deterministic(X, N, cov)
            out.append([m] * K)          # seed-free: K identical members
            FIT_COUNT += 1
        else:
            ms, _ = fit_hmm_ensemble(X, N, cov, K=K, base_seed=sd[0])
            out.append(ms)
            FIT_COUNT += K
    FIT_CACHE[key] = out
    return out


print(f'fit harness ready. R={R_SEED_SETS} DISJOINT seed sets, e.g. K=6 -> '
      f'{seed_sets(6)}')

## 5. The arms

### Interventions (protocol section "Interventions to test")

| id | axis | what changes |
|---|---|---|
| `BASELINE` | — | shipped config, `K=6`, `CONFIRM_BARS=2`, `hold='block'` |
| `S1a/S1b` | S | `ENSEMBLE_K` 6 → 12 → 24 |
| `S2` | S | drop ensemble members whose converged LL is > 25% of the LL spread below the best |
| `S3` | S | **majority-vote the per-seed emitted direction** instead of averaging direction mass |
| `S4` | S | deterministic (seed-free) initialisation |
| `L1` | L | `CONFIRM_BARS` 2 → 1 |
| `L2` | L | adaptive confirmation: 1 bar when `\|bull−bear\|` clears the fit-window 75th percentile, else 2 |
| `L3` | L | asymmetric confirmation: 1 bar to LEAVE a directional state, 2 to ENTER one |
| `L4` | L | `ESCALATION_DURING_HOLD` `'block'` → `'allow'` |

> **`L4` is reported but is expected to be a null on L, and this was written down
> before running it.** `ESCALATION_DURING_HOLD` acts only on the H↔L *intensity*
> axis; the L metric grades the *direction* axis, which that switch cannot reach.
> `L4` therefore cannot move L by construction. It is still run — because "cannot"
> deserves evidence, and because it genuinely can move W.

### Config / capacity arms (coordinator scope addition)

The hypothesis under test: **model capacity, not the S1–S4 machinery, is the
dominant driver of fit instability.** More free parameters ⇒ rougher likelihood
surface ⇒ more local optima ⇒ seeds diverge more. `rows/param` is the theoretical
predictor and is reported next to measured S so the hypothesis is *tested* rather
than assumed. The three named `CONFIGS` are taken verbatim from the master
generator; the capacity grid varies `N_STATES ∈ {3,4,5,6}` × `cov ∈ {diag, full}`,
and `B: lean-feat` supplies the feature-count axis.

Guards apply to config arms **exactly** as to intervention arms: a low-capacity
config that wins S by collapsing toward one label is VOID, not a winner.

In [ ]:
# ===========================================================================
# ARM DEFINITIONS. Every arm is a dict; nothing here runs a fit yet.
#   fit  : features / N / cov / K / init
#   lab  : kwargs handed to label_bars   (bar_dir_weight is NEVER among them)
#   post : optional ensemble post-processor (S2 pruning)
#   pol  : optional confirmation-policy factory (L2 / L3)
#   mass : optional direction-mass override  (S3)
# ===========================================================================
CFG_BY_NAME = {c['name']: c for c in CONFIGS}
BASE_CFG = CFG_BY_NAME[ADOPTED_CONFIG_NAME]
BASE_FEATURES = tuple(BASE_CFG['features'])
BASE_N, BASE_COV = BASE_CFG['N'], BASE_CFG['cov']


def arm(name, group, note, features=None, N=None, cov=None, K=ENSEMBLE_K,
        init='seeded', lab=None, post=None, pol=None, mass=False):
    return dict(name=name, group=group, note=note,
                features=tuple(features or BASE_FEATURES),
                N=N or BASE_N, cov=cov or BASE_COV, K=K, init=init,
                lab=dict(lab or {}), post=post, pol=pol, mass=mass)


# ---- S2: log-likelihood pruning -------------------------------------------
def prune_by_ll(models, X):
    lls = np.array([float(m.score(X)) for m in models])
    best, worst = lls.max(), lls.min()
    spread = best - worst
    if spread <= 0:
        return list(models), lls, len(models)
    keep = lls >= best - S2_LL_FRAC * spread
    kept = [m for m, k in zip(models, keep) if k]
    assert kept, 'LL pruning must always keep at least the best member'
    return kept, lls, len(kept)


# ---- L2 / L3: confirmation policies ---------------------------------------
def make_pol_L2(strength, thr):
    '''1-bar confirmation on strong bars, 2 otherwise. Uses ONLY bar t.'''
    def need(t, prev, cand):
        return 1 if strength[t] >= thr else CONFIRM_BARS
    return need


def make_pol_L3(_strength=None, _thr=None):
    '''Asymmetric: 1 bar to LEAVE a directional state, CONFIRM_BARS to ENTER one.

    'Extreme regime' is read at the DIRECTION level (BULL/BEAR vs SIDE), because
    that is the axis confirm_delay acts on. Leaving BULL or BEAR is cheap; taking
    a directional position from SIDE still costs the full confirmation.
    '''
    def need(t, prev, cand):
        return 1 if prev in ('BULL', 'BEAR') else CONFIRM_BARS
    return need


ARMS = [
    arm('BASELINE', 'baseline',
        f'shipped: {ADOPTED_CONFIG_NAME}, K={ENSEMBLE_K}, CONFIRM_BARS={CONFIRM_BARS}, '
        f'hold={ESCALATION_DURING_HOLD!r}'),

    # ---- S interventions --------------------------------------------------
    arm('S1a K=12', 'S', 'ENSEMBLE_K 6 -> 12', K=12),
    arm('S1b K=24', 'S', 'ENSEMBLE_K 6 -> 24', K=24),
    arm('S2 LL-prune', 'S', f'drop members > {S2_LL_FRAC:.0%} of the LL spread '
        f'below the best', post='ll_prune'),
    arm('S3 vote', 'S', 'majority-vote the per-seed EMITTED direction', mass=True),
    arm('S4 det-init', 'S', 'deterministic seed-free initialisation', init='det'),

    # ---- L interventions --------------------------------------------------
    arm('L1 confirm=1', 'L', 'CONFIRM_BARS 2 -> 1', lab=dict(confirm_bars=1)),
    arm('L2 adaptive', 'L', f'1 bar when |bull-bear| >= fit-window '
        f'p{100*(1-L2_FAST_RATE):.0f}, else 2', pol='L2'),
    arm('L3 asymmetric', 'L', 'fast to LEAVE a directional state, slow to enter',
        pol='L3'),
    arm('L4 hold=allow', 'L', "ESCALATION_DURING_HOLD 'block' -> 'allow' "
        '(intensity axis only -- expected null on L)',
        lab=dict(escalation_during_hold='allow')),
]

# ---- CONFIG / CAPACITY arms (scope addition) -------------------------------
for _c in CONFIGS:
    if _c['name'] == ADOPTED_CONFIG_NAME:
        continue                       # that IS the baseline; reused, not refitted
    ARMS.append(arm(f"CFG {_c['name']}", 'config',
                    f"named config: N={_c['N']} cov={_c['cov']} "
                    f"{len(_c['features'])} features",
                    features=_c['features'], N=_c['N'], cov=_c['cov']))

_named = {(tuple(c['features']), c['N'], c['cov']) for c in CONFIGS}
for _N in (3, 4, 5, 6):
    for _cov in ('diag', 'full'):
        if (tuple(FEATURE_COLS), _N, _cov) in _named:
            continue                   # already covered by a named config
        ARMS.append(arm(f'CAP N={_N} {_cov}', 'capacity',
                        f'capacity grid: N={_N} cov={_cov} 9 features',
                        features=FEATURE_COLS, N=_N, cov=_cov))

print(f'{len(ARMS)} arms declared before any fit:')
for _a in ARMS:
    print(f"  {_a['name']:<16} [{_a['group']:<8}] {_a['note']}")

_keys, _budget = set(), 0
for _a in ARMS:
    _k = (tuple(_a['features']), _a['N'], _a['cov'], _a['K'], R_SEED_SETS, _a['init'])
    if _k not in _keys:
        _keys.add(_k)
        _budget += (R_SEED_SETS if _a['init'] == 'det' else _a['K'] * R_SEED_SETS)
print(f'\nfit budget: {_budget} HMM fits across {len(_keys)} distinct fit sets '
      f'({len(ARMS)} arms). Labelling-side arms reuse the baseline fits; only arms '
      f'that change the FIT ITSELF refit.')

## 6. LABEL PHASE — every arm labelled, R times, **before any ZigZag exists**

This ordering is the leakage proof. When this section finishes, `ZZ_CALLS` is
still 0 and every label in the notebook has already been written.

In [ ]:
# ===========================================================================
# THE LABEL PHASE.
# ===========================================================================
assert ZZ_CALLS == 0, 'a ZigZag was computed before the label phase -- ordering broken'


def _masses_for(models, features):
    '''Direction masses as the engine computes them (used only to build the L2
    strength series -- a fit-window constant, strictly causal).'''
    b, s, r, _ = ensemble_direction_masses_by_mode(
        models, scaled_X(features), list(features), DIRECTION_EXCLUDE,
        DIRECTION_MODE, None, None, None)
    return b, s, r


def _vote_masses_factory():
    '''S3 -- replace mass-averaging with a per-seed majority VOTE.

    For every ensemble member: form that member's own direction call (argmax of
    its bull/bear/side masses, with the CONF_L low-confidence override applied to
    that member). Then the returned 'masses' are the VOTE SHARES across members.

    They are a valid probability simplex, they are permutation-invariant, their
    argmax IS the majority vote, and regime_confidence becomes the winning vote
    share -- so CONF_L keeps its meaning (a bar with no majority is SIDEWAYS).
    Everything downstream -- gates, hysteresis, prob_* columns -- is untouched.
    '''
    def voted(models, Xs, feature_subset, exclude, mode, tau, c, scale):
        models = list(models)
        votes = np.zeros((len(Xs), 3))          # bull, side, bear
        for m in models:
            b, s, r, _ = _emdbm_raw([m], Xs, feature_subset, exclude, mode, tau, c, scale)
            st = np.stack([b, r, s], axis=1)    # bull, bear, side
            conf = st.max(axis=1)
            w = st.argmax(axis=1)
            w = np.where(conf < CONF_L, 2, w)   # 2 == side
            idx = np.where(w == 0, 0, np.where(w == 1, 2, 1))   # -> bull,side,bear
            votes[np.arange(len(w)), idx] += 1.0
        votes /= len(models)
        probs0 = _filtered_posteriors(models[0], Xs)
        return votes[:, 0], votes[:, 1], votes[:, 2], probs0
    return voted


_emdbm_raw = ensemble_direction_masses_by_mode


@contextlib.contextmanager
def mass_override(fn):
    g = globals()
    old = g['ensemble_direction_masses_by_mode']
    g['ensemble_direction_masses_by_mode'] = fn
    try:
        yield
    finally:
        g['ensemble_direction_masses_by_mode'] = old


def label_one(models, a):
    '''Label the full series for ONE seed set of ONE arm.'''
    Xs = scaled_X(a['features'])
    kw = dict(a['lab'])
    kw.setdefault('dir_feats', FEAT_DF)
    ms = list(models)

    if a['post'] == 'll_prune':
        ms, lls, nkept = prune_by_ll(ms, Xs[:N_FIT])
        a.setdefault('_prune_kept', []).append(nkept)

    need_fn = None
    if a['pol'] == 'L2':
        b, s, r = _masses_for(ms, a['features'])
        strength = np.abs(b - r)
        thr = float(np.quantile(strength[:N_FIT], 1.0 - L2_FAST_RATE))
        a.setdefault('_l2_thr', []).append(thr)
        a.setdefault('_l2_rate', []).append(float(np.mean(strength >= thr)))
        need_fn = make_pol_L2(strength, thr)
    elif a['pol'] == 'L3':
        need_fn = make_pol_L3()

    stack = contextlib.ExitStack()
    with stack:
        if need_fn is not None:
            stack.enter_context(confirm_policy(need_fn))
        if a['mass']:
            stack.enter_context(mass_override(_vote_masses_factory()))
        out = label_bars(ms, Xs, DATES, nifty, TREND_RAW, N_FIT,
                         list(a['features']), **kw)
    return out['tactical_regime_state'].values.copy()


t0 = time.time()
LABELS = {}                 # arm name -> [labels_seed_set_0 .. R-1]
for a in ARMS:
    fits = get_fits(a['features'], a['N'], a['cov'], a['K'], a['init'])
    LABELS[a['name']] = [label_one(ms, a) for ms in fits]
    print(f"  labelled {a['name']:<16} K={a['K']:<3} "
          f"N={a['N']} cov={a['cov']:<4} feats={len(a['features'])}  "
          f"({time.time() - t0:5.1f}s cumulative)")

LABEL_PHASE_SECS = time.time() - t0
print(f'\nLABEL PHASE COMPLETE in {LABEL_PHASE_SECS:.1f}s   '
      f'{FIT_COUNT} HMM fits, {LABEL_CALLS} label_bars calls')
assert ZZ_CALLS == 0, 'LEAKAGE: a ZigZag existed during the label phase'
print('ASSERT OK: ZZ_CALLS == 0 -- every label in this notebook was produced '
      'before any ZigZag was computed.')

# S4 sanity: a seed-free fit must be identical across the disjoint seed sets.
_d4 = get_fits(BASE_FEATURES, BASE_N, BASE_COV, ENSEMBLE_K, 'det')
assert all(np.array_equal(LABELS['S4 det-init'][0], x) for x in LABELS['S4 det-init']), \
    'deterministic init produced seed-dependent labels'
print('ASSERT OK: S4 deterministic init gives IDENTICAL labels in all '
      f'{R_SEED_SETS} seed sets -> its S is 100% BY CONSTRUCTION.')

In [ ]:
# ===========================================================================
# EVAL PHASE BEGINS. The label phase is closed and the ZigZag may be computed.
# ===========================================================================
LABEL_PHASE_OPEN = False
print('LABEL PHASE CLOSED. Any label_bars call from here on fails the ordering '
      'check unless the phase is EXPLICITLY and loudly reopened.')

SWINGS = zigzag_swings(CLOSE, ZZ_PCT)
_moves = [100 * (CLOSE[b] - CLOSE[a]) / CLOSE[a] for a, b, _ in SWINGS]
print(f'ZigZag ({ZZ_PCT}% reversal) found {len(SWINGS)} swings over {N_BARS} bars.')
print(f'  swing length  : median {np.median([b - a for a, b, _ in SWINGS]):.0f} bars, '
      f'max {max(b - a for a, b, _ in SWINGS)}')
print(f'  swing size    : median |move| {np.median(np.abs(_moves)):.2f}%, '
      f'max {np.max(np.abs(_moves)):.2f}%')
print(f'  direction mix : {sum(1 for *_, s in SWINGS if s > 0)} up / '
      f'{sum(1 for *_, s in SWINGS if s < 0)} down')
print(f'ZZ_CALLS = {ZZ_CALLS}  -- from here on, ANY label_bars call would fail '
      'the tripwire.')


def summarise_arm(name, label_sets, note='', group='', rows_per_param=np.nan):
    '''S, L and W TOGETHER, plus occupancy, guards and the seed-set spreads.

    ONE function, used by every real arm AND by the degeneracy stubs below, so
    the guard test exercises the shipping path rather than a copy of it.
    '''
    S, M, pair_vals = metric_S(label_sets)
    L_med = [metric_L(lb, SWINGS)[0] for lb in label_sets]
    L_p75 = [metric_L(lb, SWINGS)[1] for lb in label_sets]
    unm = [metric_L(lb, SWINGS)[3] for lb in label_sets]
    Ws = [metric_W(lb) for lb in label_sets]
    occs = [occupancy_pct(lb) for lb in label_sets]
    occ = {lb: float(np.mean([o[lb] for o in occs])) for lb in REGIME_LABELS}
    W = float(np.mean(Ws))
    g = evaluate_guards(occ, W)
    return dict(name=name, group=group, note=note,
                S=S, S_min=min(pair_vals), S_max=max(pair_vals), S_matrix=M,
                L=float(np.mean(L_med)), L_lo=float(np.min(L_med)),
                L_hi=float(np.max(L_med)), L75=float(np.mean(L_p75)),
                unmatched=float(np.mean(unm)),
                W=W, W_lo=float(np.min(Ws)), W_hi=float(np.max(Ws)),
                occ=occ, guards=g, void=not guards_pass(g),
                rows_per_param=rows_per_param)


# ---- TEST 3: G4 MUST BITE on a degenerate all-SIDEWAYS labelling ------------
_stub_side = [np.full(N_BARS, 'SIDEWAYS', dtype='<U8') for _ in range(R_SEED_SETS)]
_r4 = summarise_arm('STUB all-SIDEWAYS', _stub_side, 'degeneracy stub', 'test')
print(f"\nTEST 3  degeneracy stub: S={_r4['S']:.2f}%  L={_r4['L']:.1f}  W={_r4['W']:.2f}  "
      f"SIDEWAYS={_r4['occ']['SIDEWAYS']:.1f}%")
for _k, (_p, _why) in _r4['guards'].items():
    print(f"        {_k} {'PASS' if _p else 'FAIL'}  {_why}")
assert _r4['S'] == 100.0, 'the stub should score a PERFECT S -- that is the point'
assert not _r4['guards']['G4'][0], 'G4 FAILED TO BITE on an all-SIDEWAYS labelling'
assert _r4['void'], 'the degenerate stub was not VOIDED'
print('TEST 3 PASS  a perfect S=100.00% is VOIDED by G4 (and G1/G2). '
      'S cannot be gamed by collapsing to SIDEWAYS.')

# ---- TEST 4: G3 MUST BITE on a high-whipsaw labelling -----------------------
_alt = np.array([REGIME_LABELS[i % 5] for i in range(N_BARS)], dtype='<U8')
_r3 = summarise_arm('STUB whipsaw', [_alt] * R_SEED_SETS, 'whipsaw stub', 'test')
assert not _r3['guards']['G3'][0], 'G3 FAILED TO BITE at W=100/100 bars'
print(f"TEST 4 PASS  W={_r3['W']:.1f}/100 bars is VOIDED by G3 "
      f"(max {W_GUARD_MAX}).")

## 7. Results — S, L and W together, on every row. Baseline is row 1.

In [ ]:
# ===========================================================================
# SUMMARISE EVERY ARM.
# ===========================================================================
def rpp(a):
    p = n_params(a['N'], len(a['features']), a['cov'])
    return round(N_FIT / p, 2)


RESULTS = [summarise_arm(a['name'], LABELS[a['name']], a['note'], a['group'], rpp(a))
           for a in ARMS]
RES = {r['name']: r for r in RESULTS}
BASE = RES['BASELINE']

rows = []
for r in RESULTS:
    g = r['guards']
    rows.append({
        'arm': r['name'], 'group': r['group'],
        'S %': round(r['S'], 2),
        'L med': round(r['L'], 2), 'L p75': round(r['L75'], 2),
        'W /100': round(r['W'], 2),
        'unmat': round(r['unmatched'], 1),
        'rows/par': r['rows_per_param'],
        'SIDE %': round(r['occ']['SIDEWAYS'], 1),
        'G1': 'P' if g['G1'][0] else 'F', 'G2': 'P' if g['G2'][0] else 'F',
        'G3': 'P' if g['G3'][0] else 'F', 'G4': 'P' if g['G4'][0] else 'F',
        'verdict': 'VOID' if r['void'] else 'ok',
        'dS': round(r['S'] - BASE['S'], 2),
        'dL': round(r['L'] - BASE['L'], 2),
        'dW': round(r['W'] - BASE['W'], 2),
    })
TAB = pd.DataFrame(rows)
pd.set_option('display.width', 200, 'display.max_columns', 40)
print('S = mean pairwise 5-label agreement over R=%d DISJOINT seed sets (higher better)'
      % R_SEED_SETS)
print('L = ZigZag transition lag in bars, mean over seed sets (lower better)')
print('W = label switches per 100 bars (guard: must not get worse)')
print(f'unmat = ZigZag swings NEVER labelled in the right direction (of '
      f'{len(SWINGS)}); each one already scores the FULL swing length inside L')
print('=' * 150)
print(TAB.to_string(index=False))
print('=' * 150)
print('NO COMPOSITE SCORE IS COMPUTED. The three columns are not combined.')

# ---- did the switchable interventions actually FIRE? -----------------------
# An intervention that never engages is a null for a boring reason, and that has
# to be distinguishable from one that engaged and did not help.
_armmap = {a['name']: a for a in ARMS}
_a2 = _armmap['L2 adaptive']
if '_l2_thr' in _a2:
    print(f"L2 diagnostic : threshold |bull-bear| >= "
          f"{np.mean(_a2['_l2_thr']):.4f} (fit-window p{100*(1-L2_FAST_RATE):.0f}); "
          f"the 1-bar fast path was eligible on "
          f"{100*np.mean(_a2['_l2_rate']):.1f}% of all bars")
_a3 = _armmap['S2 LL-prune']
if '_prune_kept' in _a3:
    _kept = float(np.mean(_a3['_prune_kept']))
    print(f"S2 diagnostic : kept {_kept:.2f} of {_a3['K']} ensemble members on "
          f"average (threshold {S2_LL_FRAC:.0%} of the LL spread below the best)")
    if _kept <= _a3['K'] / 2:
        print('              *** S2 IS DE-ENSEMBLING, NOT PRUNING. The across-seed LL')
        print('              spread is wide enough that a "fraction of the spread" rule')
        print(f'              discards most of the ensemble, leaving ~{_kept:.1f} members.')
        print('              That is the OPPOSITE of what S2 was meant to do, and it is')
        print('              why its S is worse, not better. The threshold is NOT being')
        print('              moved to fix this -- it was declared before the run and any')
        print('              retune would be fitting the knob to the answer. The honest')
        print('              read is that S2 as SPECIFIED is mis-specified: an absolute')
        print('              LL-gap rule, or a rule keeping at least K/2 members, would')
        print('              be the thing to pre-declare next time.')
print(f"L4 diagnostic : dL = {RES['L4 hold=allow']['L'] - BASE['L']:+.2f} bars. "
      "ESCALATION_DURING_HOLD acts on the H<->L INTENSITY axis only, so it CANNOT "
      "move a DIRECTION-lag metric. This null was predicted in section 5 before "
      "the run; dL == 0 is the evidence, not a disappointment.")

In [ ]:
# ===========================================================================
# THE FULL PAIRWISE AGREEMENT MATRICES (protocol requires the matrix, not just
# the mean) -- printed for the baseline and for every S-axis arm.
# ===========================================================================
for _n in ['BASELINE'] + [a['name'] for a in ARMS if a['group'] == 'S']:
    _r = RES[_n]
    print(f"\n{_n}   S = {_r['S']:.2f}%   pairwise range "
          f"[{_r['S_min']:.2f}, {_r['S_max']:.2f}]")
    _m = pd.DataFrame(_r['S_matrix'],
                      index=[f'set{i}' for i in range(R_SEED_SETS)],
                      columns=[f'set{i}' for i in range(R_SEED_SETS)])
    print(_m.round(2).to_string())

In [ ]:
# ===========================================================================
# HEADROOM CHECK -- REQUIRED BEFORE ANY VERDICT.
# The question: can a candidate's improvement even be distinguished from the
# seed-to-seed noise of the measurement itself?
# ===========================================================================
S_SPREAD = BASE['S_max'] - BASE['S_min']         # spread of the 6 pairwise values
L_SPREAD = BASE['L_hi'] - BASE['L_lo']           # spread of L across seed sets
W_SPREAD = BASE['W_hi'] - BASE['W_lo']           # spread of W across seed sets

print('HEADROOM CHECK (baseline, across the R=%d disjoint seed sets)' % R_SEED_SETS)
print('-' * 78)
print(f"  S : {BASE['S']:.2f}%   pairwise values span "
      f"{BASE['S_min']:.2f} -> {BASE['S_max']:.2f}   spread = {S_SPREAD:.2f} pp")
print(f"  L : {BASE['L']:.2f} bars   per-seed-set span "
      f"{BASE['L_lo']:.2f} -> {BASE['L_hi']:.2f}   spread = {L_SPREAD:.2f} bars")
print(f"  W : {BASE['W']:.2f}/100   per-seed-set span "
      f"{BASE['W_lo']:.2f} -> {BASE['W_hi']:.2f}   spread = {W_SPREAD:.2f}")
print('-' * 78)
print('RULE (pre-declared): a candidate whose improvement is SMALLER than the '
      'corresponding spread is NOISE, not a result.')


def is_noise(r):
    return (abs(r['S'] - BASE['S']) <= S_SPREAD) and (abs(r['L'] - BASE['L']) <= L_SPREAD)


_noisy = [r['name'] for r in RESULTS if r['name'] != 'BASELINE' and is_noise(r)]
print(f"\n{len(_noisy)} of {len(RESULTS) - 1} arms move BOTH S and L by less than the "
      f"baseline seed spread -> indistinguishable from noise:")
print('  ' + (', '.join(_noisy) if _noisy else '(none)'))

## 8. Combinations

Measured **after** every intervention has been measured alone, and deliberately
few: the runtime that would go into a full S×L grid is worth more spent on `R=4`
and on the config arms. The combinations tried are the best-on-S intervention
crossed with the best-on-L intervention, plus the best-on-S *config* crossed with
the best-on-L intervention.

In [ ]:
# ===========================================================================
# COMBINATIONS -- chosen from the measured single-intervention results.
# All of them REUSE already-cached fits, so this section adds no HMM fits
# unless a config arm brings its own (already-cached) fit set.
# ===========================================================================
assert ZZ_CALLS > 0, 'combination section must run in the EVAL phase'


def _best(group, key, lower_better):
    all_g = [r for r in RESULTS if r['group'] == group]
    cand = [r for r in all_g if not r['void']]
    if not cand:
        cand = all_g
        print(f'  NOTE: every {group} arm is VOID; picking on the metric anyway '
              f'(the combination inherits the VOID and is reported as such).')
    elif len(cand) < len(all_g):
        print(f'  NOTE: {len(all_g) - len(cand)} of {len(all_g)} {group} arms are '
              f'VOID and are NOT selectable, so the pick below can be worse on '
              f'{key} than a VOID arm. That is the guard doing its job, not a bug.')
    return min(cand, key=lambda r: (r[key] if lower_better else -r[key]))


_bS = _best('S', 'S', False)
_bL = _best('L', 'L', True)
_bCap = _best('capacity', 'S', False)
print(f"best S-axis intervention : {_bS['name']}  (S={_bS['S']:.2f})")
print(f"best L-axis intervention : {_bL['name']}  (L={_bL['L']:.2f})")
print(f"best capacity arm on S   : {_bCap['name']}  (S={_bCap['S']:.2f}, "
      f"rows/par={_bCap['rows_per_param']})")

_armof = {a['name']: a for a in ARMS}
COMBOS = []


def combine(fit_name, lab_name, label):
    '''Fit-side parameters come from `fit_name`, labelling-side from BOTH.'''
    A, B = _armof[fit_name], _armof[lab_name]
    c = arm(label, 'combo', f'{fit_name} + {lab_name}',
            features=A['features'], N=A['N'], cov=A['cov'],
            K=A['K'], init=A['init'],
            lab={**A['lab'], **B['lab']},
            post=A['post'] or B['post'],
            pol=A['pol'] or B['pol'],
            mass=A['mass'] or B['mass'])
    COMBOS.append(c)
    return c


combine(_bS['name'], _bL['name'], f"C1 {_bS['name']}+{_bL['name']}")
combine(_bCap['name'], _bL['name'], f"C2 {_bCap['name']}+{_bL['name']}")

# The combination arms cannot be named until the singles have been measured, and
# measuring needs the ZigZag. So the label phase is reopened -- EXPLICITLY, and
# loudly, and only for its ordering check. Object-identity, memory-aliasing and
# the w == 0.0 decree stay armed the whole time, so a ZigZag array still cannot
# reach a label here.
_zz_before = (ZZ_CALLS, len(ZZ_ARRAYS), [id(z) for z in ZZ_ARRAYS])
with reopen_label_phase('combination arms only'):
    for c in COMBOS:
        fits = get_fits(c['features'], c['N'], c['cov'], c['K'], c['init'])
        LABELS[c['name']] = [label_one(ms, c) for ms in fits]
        print(f"  labelled {c['name']}")
assert not LABEL_PHASE_OPEN
assert (ZZ_CALLS, len(ZZ_ARRAYS), [id(z) for z in ZZ_ARRAYS]) == _zz_before, \
    'the ZigZag state changed during the reopened label phase'
print('ASSERT OK: no ZigZag was computed and no ZigZag array was touched while the '
      'label phase was reopened.')

for c in COMBOS:
    RESULTS.append(summarise_arm(c['name'], LABELS[c['name']], c['note'],
                                 'combo', rpp(c)))
    ARMS.append(c)
RES = {r['name']: r for r in RESULTS}
BASE = RES['BASELINE']
_armof = {a['name']: a for a in ARMS}          # rebuilt to include the combos

_cr = pd.DataFrame([{
    'arm': r['name'], 'S %': round(r['S'], 2), 'L med': round(r['L'], 2),
    'W /100': round(r['W'], 2), 'SIDE %': round(r['occ']['SIDEWAYS'], 1),
    'guards': ' '.join(k for k, v in r['guards'].items() if not v[0]) or 'all pass',
    'verdict': 'VOID' if r['void'] else 'ok',
    'dS': round(r['S'] - BASE['S'], 2), 'dL': round(r['L'] - BASE['L'], 2),
    'dW': round(r['W'] - BASE['W'], 2),
} for r in RESULTS if r['group'] in ('baseline', 'combo')])
print()
print(_cr.to_string(index=False))

## 9. Verdict — strict dominance only

A candidate wins **only** if, versus the baseline:

`S` higher **and** `L` lower **and** `W` not worse **and** all four guards pass.

There is no composite score and no tie-break. If nothing clears that bar, the
notebook prints `NO DOMINATING CONFIGURATION` and shows the frontier — which the
protocol says is the *likely* honest outcome, because S and L oppose each other.

In [ ]:
# ===========================================================================
# VERDICT.
# ===========================================================================
print('=' * 100)
if BASE['void']:
    print('*** WARNING: THE BASELINE ITSELF IS VOID ON THIS DATA. ***')
    for k, (p, why) in BASE['guards'].items():
        if not p:
            print(f'    {k} FAIL: {why}')
    if TAC_SYNTH:
        print('    This is a SYNTHETIC series whose occupancy profile is not the real')
        print("    market's. It is a plumbing observation, NOT a finding about RegDet.")
    print('    Dominance is still evaluated below, relative to that baseline, but no')
    print('    verdict from this run is decision-grade.')
    print('=' * 100)

DOMINATING = []
for r in RESULTS:
    if r['name'] == 'BASELINE':
        continue
    if r['void']:
        continue
    if r['S'] > BASE['S'] and r['L'] < BASE['L'] and r['W'] <= BASE['W']:
        DOMINATING.append(r)

if DOMINATING:
    print('DOMINATING CONFIGURATION(S) FOUND — S up, L down, W not worse, all guards pass:')
    for r in DOMINATING:
        _n = is_noise(r)
        print(f"  {r['name']:<22} S {BASE['S']:.2f} -> {r['S']:.2f}   "
              f"L {BASE['L']:.2f} -> {r['L']:.2f}   W {BASE['W']:.2f} -> {r['W']:.2f}"
              + ('   *** INSIDE THE SEED SPREAD -> NOISE, NOT A RESULT ***' if _n else ''))
    DOMINATING = [r for r in DOMINATING if not is_noise(r)]
    if not DOMINATING:
        print('\n  ...and every one of them is inside the baseline seed spread.')
        print('  NO DOMINATING CONFIGURATION (after the headroom check).')
else:
    print('NO DOMINATING CONFIGURATION')
    print()
    print('No candidate improved S AND L without making W worse, with all guards')
    print('passing. This is the outcome the frozen protocol named as likely: lower')
    print('lag means reacting to smaller moves, which means more switches and more')
    print('sensitivity to the fit. The deliverable is the FRONTIER below, not a fix.')

# ---- the frontier: arms not dominated by any other arm on (L down, S up) ----
_live = [r for r in RESULTS if not r['void']]
FRONTIER = [r for r in _live
            if not any(o is not r and o['L'] <= r['L'] and o['S'] >= r['S']
                       and (o['L'] < r['L'] or o['S'] > r['S']) for o in _live)]
print()
print(f'PARETO FRONTIER on (L lower, S higher) over the {len(_live)} non-VOID arms:')
for r in sorted(FRONTIER, key=lambda x: x['L']):
    print(f"  L={r['L']:6.2f}  S={r['S']:6.2f}%  W={r['W']:5.2f}  "
          f"{r['name']}{'   <-- BASELINE' if r['name'] == 'BASELINE' else ''}")
if not any(r['name'] == 'BASELINE' for r in FRONTIER):
    print('  (the baseline is NOT on the frontier -- it is dominated by the arms above)')
print('=' * 100)

In [ ]:
# ===========================================================================
# THE CAPACITY HYPOTHESIS, tested rather than assumed.
# ===========================================================================
_cap = [r for r in RESULTS if r['group'] in ('capacity', 'config', 'baseline')]
_x = np.array([r['rows_per_param'] for r in _cap], dtype=float)
_y = np.array([r['S'] for r in _cap], dtype=float)
_rho, _p = spearmanr(_x, _y)
print('CAPACITY -> STABILITY')
print('-' * 78)
print(pd.DataFrame([{'arm': r['name'], 'rows/par': r['rows_per_param'],
                     'params': int(n_params(_armof[r['name']]['N'],
                                            len(_armof[r['name']]['features']),
                                            _armof[r['name']]['cov'])),
                     'S %': round(r['S'], 2), 'L med': round(r['L'], 2),
                     'W /100': round(r['W'], 2),
                     'SIDE %': round(r['occ']['SIDEWAYS'], 1),
                     'verdict': 'VOID' if r['void'] else 'ok'}
                    for r in sorted(_cap, key=lambda r: -r['rows_per_param'])
                    ]).to_string(index=False))
print('-' * 78)
print(f'Spearman rho(rows-per-param, S) = {_rho:+.3f}   p = {_p:.4f}   n = {len(_x)}')
_S_range_cap = max(r['S'] for r in _cap) - min(r['S'] for r in _cap)
_S_range_int = (max(r['S'] for r in RESULTS if r['group'] == 'S')
                - min(r['S'] for r in RESULTS if r['group'] == 'S'))
print(f'S range across CAPACITY/CONFIG arms : {_S_range_cap:.2f} pp')
print(f'S range across S1-S4 INTERVENTIONS  : {_S_range_int:.2f} pp')
print(f'baseline seed spread on S           : {S_SPREAD:.2f} pp')
if _p < 0.05 and _rho > 0:
    print('\n=> CAPACITY DRIVES STABILITY: fewer free parameters per training row goes')
    print('   with a MORE stable fit, monotonically and significantly. Config choice is')
    print('   a real stability lever, and it is measurable where forward-return ranking')
    print('   (defect 4, the "config lottery") was not.')
elif _p < 0.05 and _rho < 0:
    print('\n=> The relationship runs the WRONG WAY: more parameters per row goes with')
    print('   MORE stability here. The capacity hypothesis is not supported.')
else:
    print('\n=> NO SIGNIFICANT capacity->stability relationship at this sample size.')
    print('   The capacity hypothesis is NOT confirmed by this run. Reported as a null,')
    print('   not spun. NOTE: guards must be read alongside this -- a low-capacity arm')
    print('   that wins S by collapsing toward one label is VOID, not stable.')

## 10. Figures

Four figures. The first two are the frontiers the protocol asks for, the third
tests the capacity hypothesis visually, and the fourth is the **reference case** —
the one the lag fix has to survive being looked at.

In [ ]:
# ===========================================================================
# FIG 1 - the (L, S) frontier.  FIG 2 - the (L, W) frontier.
# ===========================================================================
SAVED_PNGS = []


def save_fig(fig, fname, dpi=130):
    fig.savefig(fname, dpi=dpi, bbox_inches='tight')
    SAVED_PNGS.append(fname)
    print(f'   saved -> {fname}')
    return fname


GROUP_STYLE = {'baseline': ('#d62728', '*', 420),
               'S':        ('#1f77b4', 'o', 90),
               'L':        ('#2ca02c', 's', 90),
               'config':   ('#9467bd', 'D', 80),
               'capacity': ('#8c564b', '^', 80),
               'combo':    ('#ff7f0e', 'P', 130)}


def frontier_panel(ax, ykey, ylabel, better_note):
    # VOID arms are drawn hollow, so a guard failure is visible on the chart and
    # cannot be mistaken for a usable point on the frontier.
    for grp, (c, mk, sz) in GROUP_STYLE.items():
        ok = [r for r in RESULTS if r['group'] == grp and not r['void']]
        vd = [r for r in RESULTS if r['group'] == grp and r['void']]
        if ok:
            ax.scatter([r['L'] for r in ok], [r[ykey] for r in ok], c=c, marker=mk,
                       s=sz, zorder=4, edgecolors='black', linewidths=0.6, label=grp)
        if vd:
            ax.scatter([r['L'] for r in vd], [r[ykey] for r in vd], facecolors='none',
                       marker=mk, s=sz, zorder=4, edgecolors=c, linewidths=1.4,
                       alpha=0.75, label=f'{grp} (VOID)')
    # ---- de-overlapped annotations -------------------------------------
    # Many arms land on IDENTICAL (L, S) coordinates -- L is a median in whole
    # bars, so ties are the norm, not the exception. Annotating them at the same
    # offset renders an unreadable pile. Coincident points are clustered and
    # their labels stacked vertically, so every arm stays legible and its
    # membership of the cluster stays obvious.
    ax.relim(); ax.autoscale_view()
    _xr = np.ptp(ax.get_xlim()) or 1.0
    _yr = np.ptp(ax.get_ylim()) or 1.0
    clusters = []
    for r in sorted(RESULTS, key=lambda r: (r['L'], -r[ykey])):
        for c in clusters:
            if (abs(r['L'] - c[0][0]) <= 0.09 * _xr
                    and abs(r[ykey] - c[0][1]) <= 0.03 * _yr):
                c[1].append(r)
                break
        else:
            clusters.append(([r['L'], r[ykey]], [r]))
    # A little room on the right so the right-most name is not clipped. The
    # y-axis is NOT stretched: S is a percentage and an axis running past 100%
    # would be nonsense. Instead a stack that sits high in the panel is written
    # DOWNWARDS, into the empty space below it.
    _x0, _x1 = ax.get_xlim()
    ax.set_xlim(_x0, _x1 + 0.12 * (_x1 - _x0))
    _mid = 0.5 * (ax.get_ylim()[0] + ax.get_ylim()[1])
    _rx0, _rx1 = ax.get_xlim()
    for (cx, cy), members in clusters:
        down = cy > _mid
        # a cluster near the right edge writes LEFTWARDS, so its stack does not
        # run off the panel or collide with whatever sits to its right
        left = (cx - _rx0) / (_rx1 - _rx0) > 0.72
        for k, r in enumerate(members):
            dy = (-1 if down else 1) * (6 + 9.0 * k)
            # names are truncated on the chart only; the results table above
            # carries every name in full
            _nm = r['name'] if len(r['name']) <= 19 else r['name'][:18] + '~'
            ax.annotate(_nm + (' [VOID]' if r['void'] else ''), (cx, cy),
                        fontsize=6.5, xytext=(-9 if left else 8, dy),
                        textcoords='offset points',
                        ha='right' if left else 'left',
                        va='top' if down else 'bottom',
                        color='#999999' if r['void'] else '#222222',
                        arrowprops=dict(arrowstyle='-', lw=0.4, color='#bbbbbb',
                                        shrinkA=0, shrinkB=2) if len(members) > 1
                        else None)
    ax.axvline(BASE['L'], color='#d62728', ls=':', lw=1.0, zorder=1)
    ax.axhline(BASE[ykey], color='#d62728', ls=':', lw=1.0, zorder=1)
    ax.set_xlabel('L  —  transition lag, median bars  (LOWER IS BETTER  <—)', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(alpha=0.25, zorder=0)
    ax.legend(fontsize=8, loc='best', ncol=2)
    ax.set_title(better_note, fontsize=10, loc='left')


fig, ax = plt.subplots(figsize=(13, 8))
frontier_panel(ax, 'S', 'S  —  mean pairwise label agreement %  (HIGHER IS BETTER)',
               'The dotted red cross is the BASELINE. The UPPER-LEFT quadrant is the '
               'only place a dominating candidate can live.')
_pf = sorted(FRONTIER, key=lambda x: x['L'])
ax.plot([r['L'] for r in _pf], [r['S'] for r in _pf], color='#444444', lw=1.2,
        ls='--', zorder=2, label='Pareto frontier (L down, S up)')
# The ONLY region where a dominating candidate can live: strictly lower L AND
# strictly higher S than the baseline. Shaded so the emptiness is the message.
_x0, _x1 = ax.get_xlim(); _y0, _y1 = ax.get_ylim()
ax.add_patch(plt.Rectangle((_x0, BASE['S']), BASE['L'] - _x0, _y1 - BASE['S'],
                           color='#2ca02c', alpha=0.07, zorder=0))
ax.annotate('DOMINATING QUADRANT\n(lower lag AND more stable)',
            ((_x0 + BASE['L']) / 2, _y1), fontsize=9, color='#2ca02c',
            ha='center', va='top', fontweight='bold')
ax.set_xlim(_x0, _x1); ax.set_ylim(_y0, _y1)
ax.legend(fontsize=8, loc='lower right', ncol=2)
fig.suptitle('FIG 1 — the (L, S) FRONTIER: transition lag vs fit stability'
             + ('  [SYNTHETIC - ILLUSTRATIVE ONLY]' if TAC_SYNTH else '  [real data]'),
             fontsize=14, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig1_frontier_L_vs_S.png')
plt.show()

fig, ax = plt.subplots(figsize=(13, 8))
frontier_panel(ax, 'W', 'W  —  switches per 100 bars  (LOWER IS BETTER — this is the '
                        'anti-gaming guard)',
               f'G3 voids anything above W = {W_GUARD_MAX}. A lag fix that cuts L by '
               'inflating W has bought nothing.')
ax.axhline(W_GUARD_MAX, color='black', ls='--', lw=1.2)
ax.annotate(f'G3 limit  W = {W_GUARD_MAX}', (ax.get_xlim()[0], W_GUARD_MAX),
            fontsize=9, va='bottom', ha='left')
fig.suptitle('FIG 2 — the (L, W) FRONTIER: does lower lag simply buy more whipsaw?'
             + ('  [SYNTHETIC - ILLUSTRATIVE ONLY]' if TAC_SYNTH else '  [real data]'),
             fontsize=14, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig2_frontier_L_vs_W.png')
plt.show()

In [ ]:
# ===========================================================================
# FIG 3 - the capacity hypothesis: S against rows-per-parameter.
# ===========================================================================
fig, (axA, axB) = plt.subplots(1, 2, figsize=(15, 6))
_cx = np.array([r['rows_per_param'] for r in _cap], float)
_cy = np.array([r['S'] for r in _cap], float)
_cv = np.array([r['void'] for r in _cap])
axA.scatter(_cx[~_cv], _cy[~_cv], s=110, c='#1f77b4', edgecolors='black',
            linewidths=0.7, zorder=3, label='guards pass')
axA.scatter(_cx[_cv], _cy[_cv], s=110, c='none', edgecolors='#d62728',
            linewidths=1.4, zorder=3, label='VOID (guard failed)')
for r in _cap:
    axA.annotate(r['name'], (r['rows_per_param'], r['S']), fontsize=7,
                 xytext=(5, 4), textcoords='offset points')
if len(_cx) >= 2 and np.ptp(_cx) > 0:
    _b = np.polyfit(_cx, _cy, 1)
    _xx = np.linspace(_cx.min(), _cx.max(), 50)
    axA.plot(_xx, np.polyval(_b, _xx), color='#888888', ls='--', lw=1.2,
             label=f'least squares (slope {_b[0]:+.3f} pp per row/param)')
axA.set_xlabel('rows per free parameter  (HIGHER = lower capacity = the hypothesis '
               'says MORE stable)', fontsize=9)
axA.set_ylabel('S — mean pairwise label agreement %', fontsize=10)
axA.set_title(f'A. capacity vs stability   Spearman rho = {_rho:+.3f}  (p = {_p:.4f})',
              fontsize=11, loc='left')
axA.grid(alpha=0.25)
axA.legend(fontsize=8)

_w = 0.35
_ix = np.arange(len(_cap))
_ord = sorted(range(len(_cap)), key=lambda i: -_cap[i]['rows_per_param'])
axB.bar(_ix - _w / 2, [_cap[i]['S'] for i in _ord], _w, color='#1f77b4', label='S %')
axB2 = axB.twinx()
axB2.bar(_ix + _w / 2, [_cap[i]['L'] for i in _ord], _w, color='#2ca02c', label='L med')
axB.set_xticks(_ix)
axB.set_xticklabels([_cap[i]['name'] for i in _ord], rotation=40, ha='right', fontsize=7)
axB.set_ylabel('S %', color='#1f77b4')
axB2.set_ylabel('L median bars', color='#2ca02c')
axB.axhline(BASE['S'], color='#d62728', ls=':', lw=1.2)
# S sits in a narrow band near 100%; a zero-anchored axis would hide every
# difference the panel exists to show. Limits are set to the DATA range.
_slo = min(r['S'] for r in _cap)
axB.set_ylim(_slo - 0.05 * (100 - _slo) - 0.5, 100.6)
axB2.set_ylim(0, max(r['L'] for r in _cap) * 1.25)
axB.set_title('B. S (blue, left — NOTE: axis is NOT zero-anchored) and L (green, '
              'right), ordered by rows/param.\n   dotted red = baseline S; '
              'RED tick labels = VOID (a guard failed)', fontsize=9, loc='left')
for _t, i in zip(axB.get_xticklabels(), _ord):
    if _cap[i]['void']:
        _t.set_color('#d62728')
        _t.set_fontweight('bold')
fig.suptitle('FIG 3 — does MODEL CAPACITY drive fit instability?'
             + ('  [SYNTHETIC - ILLUSTRATIVE ONLY]' if TAC_SYNTH else '  [real data]'),
             fontsize=14, fontweight='bold')
fig.tight_layout()
save_fig(fig, 'fig3_capacity_vs_stability.png')
plt.show()

In [ ]:
# ===========================================================================
# FIG 4 - THE REFERENCE CASE: the May-2025 V-bottom window, baseline vs the best
# lag candidate, in the MASTER NOTEBOOK'S regime-background style:
#   black price line, FULL-HEIGHT regime bands, TIGHT y-limits (NOT zero-anchored),
#   asserted EQUAL across panels.
# ===========================================================================
REF_START, REF_END = pd.Timestamp('2025-05-05'), pd.Timestamp('2025-05-26')

_m = (DATES >= REF_START) & (DATES <= REF_END)
if _m.sum() < 6:
    # The reference window is outside this data span (can happen on the synthetic
    # fallback or a short yfinance window). Fall back to the LARGEST up-swing, and
    # say so on the figure rather than silently plotting something else.
    _up = max([s for s in SWINGS if s[2] > 0],
              key=lambda s: (CLOSE[s[1]] - CLOSE[s[0]]) / CLOSE[s[0]])
    _a, _b = max(0, _up[0] - 12), min(N_BARS - 1, _up[1] + 12)
    _m = np.zeros(N_BARS, bool); _m[_a:_b + 1] = True
    REF_NOTE = (f'REFERENCE WINDOW {REF_START:%Y-%m-%d} -> {REF_END:%Y-%m-%d} IS NOT IN '
                f'THIS DATA SPAN — substituted the largest up-swing in the series')
else:
    REF_NOTE = f'REFERENCE CASE: {REF_START:%Y-%m-%d} -> {REF_END:%Y-%m-%d}'

REF_IDX = DATES[_m]
REF_CLOSE = CLOSE[_m]
REF_I0 = int(np.flatnonzero(_m)[0])
_ref_move = 100 * (REF_CLOSE[-1] - REF_CLOSE.min()) / REF_CLOSE.min()

# the best LAG candidate = lowest L among non-VOID L-axis and combo arms
_lagcands = [r for r in RESULTS if r['group'] in ('L', 'combo') and not r['void']]
if not _lagcands:
    _lagcands = [r for r in RESULTS if r['group'] in ('L', 'combo')]
    print('NOTE: every lag candidate is VOID; the chart shows the lowest-L one anyway, '
          'marked VOID.')
BEST_LAG = min(_lagcands, key=lambda r: r['L'])
print(f"reference-case comparison: BASELINE  vs  {BEST_LAG['name']}"
      f"{' [VOID]' if BEST_LAG['void'] else ''}")

def shade_regimes_contiguous(ax, series, alpha=0.35):
    '''Regime bands that BUTT UP against each other, with no unpainted gaps.

    The master's `regime_blocks` ends a band on the LAST BAR of its run, so the
    interval between one run's last bar and the next run's first bar is left
    white. Over 2900 bars that is invisible; on a 45-bar zoom that crosses a
    weekend it is a conspicuous white stripe, and it reads as "no regime here",
    which is false -- the regime holds across the gap.

    Each band is therefore extended to the START of the next band. The PAINTER is
    still the master's `shade_bands`, unmodified, so colour, alpha, full-height
    geometry and the autolim fix are all inherited rather than reimplemented.
    '''
    vals, idx = np.asarray(series.values), series.index
    spans, start = [], 0
    for i in range(1, len(vals)):
        if vals[i] != vals[i - 1]:
            spans.append((vals[start], idx[start], idx[i]))
            start = i
    spans.append((vals[start], idx[start], idx[-1]))
    shade_bands(ax, spans, alpha=alpha)


_panels = [('BASELINE', BASE), (BEST_LAG['name'], BEST_LAG)]
fig, axes = plt.subplots(len(_panels), 1, figsize=(16, 9), sharex=True)
_ylim = None
for ax, (nm, r) in zip(axes, _panels):
    lab = pd.Series(LABELS[nm][0], index=DATES)[_m]      # seed set 0, both panels
    shade_regimes_contiguous(ax, lab, alpha=0.35)
    ax.plot(REF_IDX, REF_CLOSE, color='black', linewidth=1.7, zorder=4)
    set_price_ylim(ax, REF_CLOSE, pad=0.03, tag=f'FIG4 {nm}')
    if _ylim is None:
        _ylim = ax.get_ylim()
    ax.set_ylim(*_ylim)                    # EXPLICIT and EQUAL across panels
    ax.set_xlim(REF_IDX[0], REF_IDX[-1])
    ax.set_ylabel('Nifty close', fontsize=9)

    # THE LAG, drawn: for every ZigZag swing starting inside the window, mark the
    # swing start, mark the first correctly-directed emitted bar, and span the
    # gap. That span IS L for that swing -- this is the picture the metric is a
    # summary of, so the two can be checked against each other by eye.
    _d = emitted_direction(lab.values)
    _shown = []
    for i0, i1, sgn in SWINGS:
        if not _m[i0]:
            continue
        c = '#0033cc' if sgn > 0 else '#cc0033'
        t0 = DATES[i0]
        ax.axvline(t0, color=c, ls='--', lw=1.3, zorder=6)
        _loc = np.flatnonzero(np.asarray(lab.index) == np.datetime64(t0))
        _off = int(_loc[0]) if _loc.size else 0
        _hit = np.flatnonzero(_d[_off:] == sgn)
        _y = _ylim[0] + 0.93 * (_ylim[1] - _ylim[0]) - 0.06 * (_ylim[1] - _ylim[0]) * len(_shown)
        if _hit.size:
            _lag = int(_hit[0])
            t1 = lab.index[_off + _lag]
            ax.annotate('', xy=(t1, _y), xytext=(t0, _y),
                        arrowprops=dict(arrowstyle='<->', color=c, lw=1.5))
            ax.plot([t1], [REF_CLOSE[_off + _lag]], marker='o', ms=9, mfc='none',
                    mec=c, mew=2.0, zorder=7)
            ax.annotate(f"swing {'UP' if sgn > 0 else 'DOWN'} — L = {_lag} bars",
                        ((t0 + (t1 - t0) / 2) if _lag else t0, _y), fontsize=9,
                        color=c, ha='center', va='bottom', fontweight='bold')
        else:
            ax.annotate(f"swing {'UP' if sgn > 0 else 'DOWN'} — NEVER matched",
                        (t0, _y), fontsize=9, color=c, ha='left', va='bottom',
                        fontweight='bold')
        _shown.append(i0)

    _first = np.flatnonzero(_d > 0)
    _ft = (f'first BULL bar in window: {lab.index[_first[0]]:%Y-%m-%d %H:%M}'
           if _first.size else 'NEVER labelled BULL in this window')
    ax.set_title(f"{nm}{'  [VOID — a guard failed]' if r['void'] else ''}   "
                 f"S={r['S']:.2f}%   L={r['L']:.2f} bars (whole series)   "
                 f"W={r['W']:.2f}   |   {_ft}",
                 fontsize=11, fontweight='bold', loc='left')

# The regime key goes ABOVE the panels, not inside them: at this zoom an in-axes
# legend covers the price line it is supposed to be explaining.
fig.legend(handles=[mpatches.Patch(color=REGIME_COLORS[r], alpha=0.7, label=r)
                    for r in REGIME_LABELS],
           loc='upper center', bbox_to_anchor=(0.5, 0.925), ncol=5, fontsize=9,
           frameon=False)
axes[-1].set_xlabel('date', fontsize=10)
assert axes[0].get_ylim() == axes[1].get_ylim(), 'panel y-limits differ'
assert axes[0].get_ylim()[0] > 0, 'y-axis is zero-anchored -- tight limits required'
fig.suptitle('FIG 4 — ' + REF_NOTE
             + f'   ({len(REF_IDX)} bars, low->close +{_ref_move:.2f}%)'
             + ('\n[SYNTHETIC - ILLUSTRATIVE ONLY — this is NOT the real May-2025 '
                'V-bottom]' if TAC_SYNTH else '')
             + '\nsame data, same fit seeds, same y-limits — only the labelling differs',
             fontsize=13, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, 0.905))
print(f'ASSERT OK: both panels share y-limits {axes[0].get_ylim()} '
      f'(tight, NOT zero-anchored)')
save_fig(fig, 'fig4_reference_case_may2025.png')
plt.show()

## 11. Verification block and config hand-off

In [ ]:
# ===========================================================================
# FINAL VERIFICATION -- the leakage tripwire, the decrees, the figure inventory.
# ===========================================================================
print('=' * 90)
print('VERIFICATION')
print('=' * 90)
print(f'  label_bars calls (all guarded)      : {LABEL_CALLS}')
print(f'  HMM fits                            : {FIT_COUNT}')
print(f'  ZigZag computations                 : {ZZ_CALLS}')
print(f'  arms measured                       : {len(RESULTS)}')
print(f'  seed sets per arm (R)               : {R_SEED_SETS}  {seed_sets(ENSEMBLE_K)}')
assert BAR_DIR_WEIGHT == 0.0
print(f'  BAR_DIR_WEIGHT                      : {BAR_DIR_WEIGHT}  (asserted on every call)')

# The leakage assertion, restated as an explicit end-of-run check.
try:
    label_bars(None)
except AssertionError as e:
    print(f'  ZigZag leakage tripwire (live test) : ARMED -> {str(e)[:60]}...')
else:
    raise AssertionError('the leakage tripwire did NOT fire after the eval phase')

print(f'  PNGs written                        : {len(SAVED_PNGS)}')
for f in SAVED_PNGS:
    print(f'      {f}')
print(f'  total runtime                       : {time.time() - T_START:.1f}s')
print('=' * 90)
if TAC_SYNTH:
    print('*** SYNTHETIC - ILLUSTRATIVE ONLY. No verdict above is decision-grade. ***')
    print('*** Re-run on Kaggle with real yfinance data.                          ***')
    print('=' * 90)

In [ ]:
# ===========================================================================
# CONFIG HAND-OFF BLOCK -- pasteable Python recording every parameter of the
# arms that matter, so a later decision can be folded into the master.
# ===========================================================================
def _handoff(r):
    a = _armof[r['name']]
    return (f"    {r['name']!r}: dict(\n"
            f"        # S={r['S']:.2f}%  L_med={r['L']:.2f}  L_p75={r['L75']:.2f}  "
            f"W={r['W']:.2f}  rows_per_param={r['rows_per_param']}\n"
            f"        # guards: " + ' '.join(f"{k}={'PASS' if v[0] else 'FAIL'}"
                                             for k, v in r['guards'].items())
            + f"   -> {'VOID' if r['void'] else 'ok'}\n"
            f"        N_STATES={a['N']}, COVARIANCE_TYPE={a['cov']!r},\n"
            f"        FEATURES={list(a['features'])!r},\n"
            f"        ENSEMBLE_K={a['K']}, INIT={a['init']!r},\n"
            f"        CONFIRM_BARS={a['lab'].get('confirm_bars', CONFIRM_BARS)},\n"
            f"        ESCALATION_DURING_HOLD="
            f"{a['lab'].get('escalation_during_hold', ESCALATION_DURING_HOLD)!r},\n"
            f"        CONFIRM_POLICY={a['pol']!r}, ENSEMBLE_POST={a['post']!r},\n"
            f"        DIRECTION_VOTE={a['mass']!r},\n"
            f"    ),")


_show = [BASE, BEST_LAG] + ([r for r in FRONTIER if r['name'] not in
                             ('BASELINE', BEST_LAG['name'])])
_seen, _uniq = set(), []
for r in _show:
    if r['name'] not in _seen:
        _seen.add(r['name']); _uniq.append(r)

print('# ' + '=' * 78)
print('# REGDET V1.1 -- STABILITY/LAG CONFIG HAND-OFF BLOCK')
print(f'# generated {pd.Timestamp.now():%Y-%m-%d %H:%M}   '
      f'data = {"SYNTHETIC (ILLUSTRATIVE ONLY)" if TAC_SYNTH else "REAL yfinance 2h"}')
print(f'# bars={N_BARS}  fit_window={N_FIT}  R={R_SEED_SETS} disjoint seed sets  '
      f'ZigZag={ZZ_PCT}%')
print('#')
print('# FIXED BY DECREE FOR ALL ARMS BELOW:')
print(f'BAR_DIR_WEIGHT = {W_FIXED}          # settled; not swept in this notebook')
print(f'INTENSITY_MODE = {INTENSITY_MODE!r}')
print(f'DIRECTION_MODE = {DIRECTION_MODE!r}')
print(f'H_TARGET_RATE, H_EXIT_SLACK = {H_TARGET_RATE}, {H_EXIT_SLACK}')
print(f'DIRECTION_EXCLUDE = {DIRECTION_EXCLUDE!r}')
print(f'BASE_SEED = {BASE_SEED}')
print()
print('# VERDICT: ' + ('dominating -> ' + ', '.join(r['name'] for r in DOMINATING)
                       if DOMINATING else 'NO DOMINATING CONFIGURATION'))
print(f"# HEADROOM: baseline seed spread  S {S_SPREAD:.2f} pp   L {L_SPREAD:.2f} bars"
      f"   W {W_SPREAD:.2f}")
print('STABILITY_LAG_ARMS = {')
for r in _uniq:
    print(_handoff(r))
print('}')
print('# ' + '=' * 78)

## 12. What this notebook does and does not establish

**Does.** It measures S, L and W on one shared, cached set of fits with `R = 4`
disjoint seed sets; it proves no ZigZag output can reach a label; it proves the
degeneracy guard actually bites; and it reports every arm's three numbers together
with its guard verdict.

**Does not.** It does not compute a forward return, a Sharpe, or any economic
metric — deliberately, because ranking configs on those is exactly the
noise-dominated procedure that produced the "config lottery" (defect 4). It does
not tune a threshold to make anything pass. It does not combine S, L and W into a
score. And on a sandbox run it does not establish anything at all about the real
market: the synthetic fallback is a plumbing test.

**The decision it is meant to support** is a choice of point on a frontier, made
by a human looking at figure 4 and asking whether the lag actually improved enough
to be worth what it cost in stability and whipsaw.